# Weekday and Weekend Differences in Shared Mobility Patterns

## A Bike-Share Case Study Using 2025 Chicago Divvy Data

This notebook is the proposal-facing summary layer. The reproducible data cleaning and analysis pipeline stays in `src/`, while this notebook reads the generated CSV tables and figures from `outputs/` and `figures/`.

In [ ]:
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUTPUTS = ROOT / "outputs"
FIGURES = ROOT / "figures"

pd.options.display.float_format = "{:,.3f}".format

def read_output_csv(filename):
    path = OUTPUTS / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing output file: {path}. Run the src pipeline first.")
    return pd.read_csv(path)

def show_png(filename, figsize=(8, 5)):
    path = FIGURES / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing figure file: {path}. Run the src pipeline first.")
    image = mpimg.imread(path)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(image)
    ax.axis("off")
    plt.show()

## Research Question

How do weekday and weekend Divvy trips in Chicago differ in trip duration, straight-line distance, spatial concentration, and the types of urban services accessed near trip end locations during 2025?

## Proposal Objective and Rubric Fit

This project analyzes weekday and weekend differences in Chicago Divvy bike-share trips during 2025. The weekday/weekend comparison is used as a starting point for understanding how shared mobility patterns change across different social and temporal contexts.

The project fits the proposal requirements because it uses a full-year mobility dataset, applies reproducible data cleaning, reports baseline statistics, examines full trip duration and distance distributions, and includes spatial heatmaps. In response to the baseline feedback, I added log-scaled distribution plots and CCDF plots for trip duration and straight-line distance. In response to the spatial feedback, I added a heatmap with an OpenStreetMap/Carto basemap. To add more depth to the research question, I also enriched sampled trip end locations with OpenStreetMap service categories, allowing weekday and weekend destinations to be compared by nearby urban service environments.

## Hypotheses / Expected Patterns

I expect weekend trips to have longer durations and slightly longer straight-line distances than weekday trips. Because trip duration and distance are likely to be right-skewed, I compare medians, interquartile ranges, log-scaled distributions, and CCDF plots instead of relying only on mean values.

I also expect weekday trips to be more concentrated around dense urban service environments such as transit, office, food and drink, and retail areas. Weekend trips are expected to be more dispersed and relatively more connected to tourism, recreation, lakefront, park, or less service-dense destination environments.

## Data and Reproducible Pipeline

The project uses monthly 2025 Chicago Divvy trip CSV files. The source data is cleaned and summarized with Python scripts in `src/`.

To reproduce this notebook, first run the pipeline scripts from the project root in this order, then run this notebook:

1. `01_combine_and_clean.py`
2. `02_eda_trip_patterns.py`
3. `03_dwell_proxy.py`
4. `04_spatial_heatmaps.py`
5. `06_distribution_fits.py`
6. `07_basemap_heatmap.py`
7. `08_prepare_end_location_sample.py`
8. `09_osm_destination_services.py`

`05_optional_weather_merge.py` is optional and is not required for the revised core analysis.

In [ ]:
cleaning_summary = read_output_csv("cleaning_summary.csv")
weekday_weekend_counts = read_output_csv("weekday_weekend_counts.csv")
monthly_counts = read_output_csv("monthly_counts_by_day_type.csv")
trip_summary = read_output_csv("trip_summary_by_day_type.csv")
dwell_summary = read_output_csv("dwell_proxy_summary_by_day_type.csv")

## Dataset Scope and Coordinate System

The dataset comes from the official Chicago Divvy monthly trip records. Each row represents one observed trip with start/end times, start/end station fields, start/end coordinates, bike type, and rider type.

The native coordinate reference system is longitude/latitude in **WGS84 / EPSG:4326**. The current pipeline keeps coordinates in this CRS for mapping and uses the haversine formula to estimate straight-line trip length in kilometers. This distance is useful for comparison, but it is not the actual street-network route distance.


In [ ]:
final_rows = int(cleaning_summary.loc[cleaning_summary["stage"] == "final rows", "rows"].iloc[0])
weekday_count = int(weekday_weekend_counts.loc[weekday_weekend_counts["day_type"] == "weekday", "trip_count"].iloc[0])
weekend_count = int(weekday_weekend_counts.loc[weekday_weekend_counts["day_type"] == "weekend", "trip_count"].iloc[0])

dataset_scope = pd.DataFrame(
    [
        {"item": "Cleaned trip records", "value": f"{final_rows:,}"},
        {"item": "Time span", "value": "2025-01-01 to 2025-12-31"},
        {"item": "Start-coordinate bounds", "value": "lat 41.65-42.07, lng -87.89--87.52"},
        {"item": "Native CRS", "value": "WGS84 longitude/latitude (EPSG:4326)"},
        {"item": "Weekday trips", "value": f"{weekday_count:,}"},
        {"item": "Weekend trips", "value": f"{weekend_count:,}"},
        {"item": "Primary strata", "value": "weekday vs weekend"},
    ]
)

display(dataset_scope)


## Proposal-Facing Summary Table

For the two-page written proposal, use a compact table instead of the full cleaning log. The full cleaning table remains below for reproducibility, but the proposal only needs the final dataset size and the weekday/weekend split.


In [ ]:
proposal_summary_table = pd.DataFrame(
    [
        {"item": "Original rows", "value": f"{int(cleaning_summary.loc[cleaning_summary['stage'] == 'original rows', 'rows'].iloc[0]):,}"},
        {"item": "Final cleaned rows", "value": f"{final_rows:,}"},
        {"item": "Weekday trips", "value": f"{weekday_count:,}"},
        {"item": "Weekend trips", "value": f"{weekend_count:,}"},
        {"item": "Primary comparison", "value": "weekday vs weekend"},
    ]
)

display(proposal_summary_table)


## Cleaning Summary

The cleaning pipeline removes invalid timestamps, trips outside the 2025 analysis year, missing or invalid coordinates, invalid durations, and unrealistic straight-line distances.

In [ ]:
display(cleaning_summary)

## Weekday vs Weekend Trip Counts

This table gives the overall sample balance between weekday and weekend trips.

In [ ]:
counts_for_display = weekday_weekend_counts.copy()
counts_for_display["share_percent"] = counts_for_display["share"] * 100
display(counts_for_display)

## Monthly Coverage

The monthly count figure shows that the analysis covers the full year instead of a single seasonal snapshot.

In [ ]:
monthly_pivot = monthly_counts.pivot(index="month_name", columns="day_type", values="trip_count")
month_order = monthly_counts.drop_duplicates("month").sort_values("month")["month_name"]
monthly_pivot = monthly_pivot.reindex(month_order)
display(monthly_pivot)
show_png("monthly_trip_counts_by_day_type.png", figsize=(9, 5))

## Exploratory Data Analysis and Baseline Statistics

The baseline EDA shows that the cleaned Divvy 2025 dataset is suitable for comparing weekday and weekend mobility patterns. The main temporal split is based on the trip start time, with trips classified as either weekday or weekend.

The first comparison focuses on trip duration and straight-line trip distance. Weekend trips have longer typical durations and slightly longer straight-line distances than weekday trips. However, the distributions are strongly right-skewed, so the analysis does not rely only on means or medians. I also examine log-scaled histograms and CCDF plots to show the full distributional structure. These plots show that weekend trips have a heavier upper tail, especially for trip duration, meaning that longer trips are more common on weekends.

The spatial baseline analysis uses start-location heatmaps. Because weekday trips are more numerous than weekend trips, normalized density is used to compare relative spatial concentration. I also include an interactive heatmap with an OpenStreetMap/Carto basemap so that the hotspots can be interpreted in relation to downtown Chicago, the lakefront, parks, streets, and surrounding neighborhoods.

To add more behavioral depth, I enrich sampled trip end locations with OpenStreetMap service categories. For each sampled endpoint, I identify the nearest mapped OSM service node within 250 meters and classify it into broad categories such as transit, food and drink, retail, tourism, office, health, recreation, and education. This allows the project to compare not only where weekday and weekend trips occur, but also what types of urban service environments are near their destinations.

The OSM results suggest that weekday endpoints are more strongly associated with dense service environments such as food and drink, transit, office, and retail. Weekend endpoints are more likely to fall outside a mapped service node within 250 meters and are slightly more associated with tourism. This supports the interpretation that weekday trips are more connected to routine urban activity areas, while weekend trips are more dispersed and somewhat more connected to leisure or tourism-oriented destinations.


## Trip Duration and Trip Length

The summary statistics use the cleaned full dataset. The boxplots hide extreme visual outliers by limiting the displayed y-axis range, but the table below is based on all filtered records.

In [ ]:
display(trip_summary)

In [ ]:
figure_names = ["trip_length_boxplot.png", "trip_duration_boxplot.png"]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, filename in zip(axes, figure_names):
    image = mpimg.imread(FIGURES / filename)
    ax.imshow(image)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Distributional structure of trip duration and distance

To respond to the baseline-distribution concern, I examined the full distributions of trip duration and straight-line trip distance instead of relying only on median values. Both variables are strongly right-skewed and have long tails, meaning that most trips are short but a smaller number of trips are much longer.

I use log-log histograms and complementary cumulative distribution functions (CCDFs) to compare weekday and weekend distributions. The CCDF plots show that weekend trips have a heavier upper tail than weekday trips, especially for trip duration. This supports the earlier median-based finding that weekend trips tend to last longer, but it also shows that the difference is visible across the distribution, not only at the median.

I also estimate simple lognormal parameters and power-law tail parameters as descriptive statistics. These fitted values are used to describe the statistical structure of the data, while the visual evidence suggests that the distributions are long-tailed rather than normally distributed.

In [ ]:
distribution_fit_summary = read_output_csv("distribution_fit_summary.csv")
display(distribution_fit_summary)

In [ ]:
show_png("trip_duration_distribution_loglog.png", figsize=(8, 5))
show_png("trip_distance_distribution_loglog.png", figsize=(8, 5))
show_png("trip_duration_ccdf.png", figsize=(8, 5))
show_png("trip_distance_ccdf.png", figsize=(8, 5))

## Baseline Distribution Form

The baseline movement metrics are right-skewed and long-tailed. The means are higher than the medians for both trip duration and straight-line trip length, so the analysis emphasizes medians, interquartile ranges, and distributional plots rather than relying only on averages.

The trip summary shows that weekend trips have longer typical durations and slightly longer straight-line distances than weekday trips. The distribution-fit results confirm the same pattern after filtering to positive duration and positive distance records. In the fitted distribution table, the weekend median duration is about 11.14 minutes compared with 9.21 minutes for weekdays, and the weekend median straight-line distance is about 1.84 km compared with 1.69 km for weekdays.

The positive skew values and CCDF plots show that both variables have long upper tails. This means that most trips are short, but a smaller number of trips are much longer, especially on weekends.

## Spatial Start-location Density

The side-by-side heatmap shows whether weekday and weekend trips concentrate in different parts of the city. The static figure uses longitude and latitude bins for reproducibility. To address the need for geographic reference, I also include an interactive Folium heatmap with a Carto/OpenStreetMap basemap below.

In [ ]:
show_png("heatmap_start_locations_weekday_weekend.png", figsize=(11, 6))

## Normalized Weekday-Weekend Difference Heatmap

The side-by-side heatmap shows where trip starts are dense, but raw counts can be misleading because weekday trips are more numerous than weekend trips. The difference heatmap below normalizes each day type separately, then plots:

`normalized weekend start density - normalized weekday start density`

Positive cells are more weekend-leaning after normalization. Negative cells are more weekday-leaning. This is a stronger final-project comparison figure than raw side-by-side counts because it asks where the spatial pattern changes, not just where total trip volume is high.


In [ ]:
def load_start_locations_for_difference(cleaned_path: Path) -> pd.DataFrame:
    if not cleaned_path.exists():
        raise FileNotFoundError(
            f"Missing cleaned data file: {cleaned_path}. Run src/01_combine_and_clean.py first."
        )
    columns = ["day_type", "start_lat", "start_lng"]
    df = pd.read_csv(cleaned_path, usecols=columns)
    return df.dropna(subset=columns)


def plot_normalized_difference_heatmap(df: pd.DataFrame, output_path: Path, bins: int = 180) -> None:
    plot_df = df.loc[df["day_type"].isin(["weekday", "weekend"])].copy()
    lng_min, lng_max = plot_df["start_lng"].quantile([0.005, 0.995])
    lat_min, lat_max = plot_df["start_lat"].quantile([0.005, 0.995])
    plot_range = [[lng_min, lng_max], [lat_min, lat_max]]

    weekday = plot_df.loc[plot_df["day_type"] == "weekday"]
    weekend = plot_df.loc[plot_df["day_type"] == "weekend"]

    weekday_counts, x_edges, y_edges = np.histogram2d(
        weekday["start_lng"], weekday["start_lat"], bins=bins, range=plot_range
    )
    weekend_counts, _, _ = np.histogram2d(
        weekend["start_lng"], weekend["start_lat"], bins=[x_edges, y_edges]
    )

    weekday_density = weekday_counts / weekday_counts.sum()
    weekend_density = weekend_counts / weekend_counts.sum()
    difference = weekend_density - weekday_density

    limit = np.nanpercentile(np.abs(difference), 99)
    if limit == 0:
        limit = np.nanmax(np.abs(difference))

    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    image = ax.imshow(
        difference.T,
        origin="lower",
        extent=[lng_min, lng_max, lat_min, lat_max],
        cmap="RdBu_r",
        vmin=-limit,
        vmax=limit,
        aspect="auto",
    )
    ax.set_title("Normalized Weekend - Weekday Start-location Density")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    colorbar = fig.colorbar(image, ax=ax, shrink=0.85)
    colorbar.set_label("Density difference per spatial bin")
    ax.text(
        0.02,
        0.02,
        "Red = weekend-leaning; blue = weekday-leaning",
        transform=ax.transAxes,
        fontsize=9,
        bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
    )
    fig.tight_layout()
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.show()


difference_heatmap_path = FIGURES / "weekday_weekend_difference_heatmap.png"
start_locations_for_difference = load_start_locations_for_difference(ROOT / "data" / "processed" / "divvy_2025_cleaned.csv")
plot_normalized_difference_heatmap(start_locations_for_difference, difference_heatmap_path)
print(f"Saved normalized difference heatmap to {difference_heatmap_path}")


## Interactive Heatmap with Basemap

The interactive Folium map adds a Carto/OpenStreetMap basemap under the weekday and weekend heatmap layers. This provides geographic context for interpreting spatial density in relation to downtown Chicago, the lakefront, parks, streets, and surrounding neighborhoods.

In [ ]:
import matplotlib as mpl
import numpy as np
import folium
from folium.plugins import HeatMap


def from_cmap(
    cmap: str,
    alpha: float = 1.0,
    reverse: bool = False,
    num: int = 100,
) -> dict[float, str]:
    cmap_obj = mpl.colormaps[cmap]
    pctiles = np.linspace(0, 1, num=num + 1)[1:]

    def rgba_to_string(r: float, g: float, b: float, a: float) -> str:
        return f"rgba({int(r * 255)}, {int(g * 255)}, {int(b * 255)}, {a:.3f})"

    gradient = {}
    for pct in pctiles:
        value = 1 - pct if reverse else pct
        gradient[float(pct)] = rgba_to_string(*cmap_obj(value, alpha=alpha))
    return gradient


def sample_start_locations(cleaned_path: Path, max_points_per_day_type: int = 2500) -> pd.DataFrame:
    if not cleaned_path.exists():
        raise FileNotFoundError(
            f"Missing cleaned data file: {cleaned_path}. Run src/01_combine_and_clean.py first."
        )

    columns = ["day_type", "start_lat", "start_lng"]
    chunk_samples = []
    per_chunk_target = max(100, max_points_per_day_type // 12)

    for chunk_number, chunk in enumerate(pd.read_csv(cleaned_path, usecols=columns, chunksize=250_000)):
        chunk = chunk.dropna(subset=columns)
        for day_type in ["weekday", "weekend"]:
            group = chunk.loc[chunk["day_type"] == day_type]
            if group.empty:
                continue
            take = min(len(group), per_chunk_target)
            chunk_samples.append(group.sample(n=take, random_state=42 + chunk_number))

    if not chunk_samples:
        raise ValueError("No start-location records were available for the interactive heatmap.")

    sampled = pd.concat(chunk_samples, ignore_index=True)
    balanced_samples = []
    for day_type in ["weekday", "weekend"]:
        group = sampled.loc[sampled["day_type"] == day_type]
        take = min(len(group), max_points_per_day_type)
        balanced_samples.append(group.sample(n=take, random_state=42))

    return pd.concat(balanced_samples, ignore_index=True)


cleaned_data_path = ROOT / "data" / "processed" / "divvy_2025_cleaned.csv"
interactive_heatmap_path = FIGURES / "interactive_start_location_heatmap.html"
heatmap_points = sample_start_locations(cleaned_data_path)

m = folium.Map(
    location=[heatmap_points["start_lat"].mean(), heatmap_points["start_lng"].mean()],
    zoom_start=11,
    tiles="CartoDB positron",
)

for day_type, label, cmap_name, show_layer in [
    ("weekday", "Weekday start locations", "viridis", True),
    ("weekend", "Weekend start locations", "magma", False),
]:
    layer_points = heatmap_points.loc[heatmap_points["day_type"] == day_type, ["start_lat", "start_lng"]]
    layer = folium.FeatureGroup(name=label, show=show_layer)
    HeatMap(
        data=layer_points.values.tolist(),
        radius=9,
        blur=13,
        min_opacity=0.25,
        max_zoom=13,
        gradient=from_cmap(cmap_name, alpha=0.85, num=16),
    ).add_to(layer)
    layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.save(interactive_heatmap_path)
print(f"Saved interactive heatmap to {interactive_heatmap_path}")
m


### OpenStreetMap destination service analysis

To add behavioral depth beyond the weekday/weekend comparison, I enriched sampled Divvy end locations with nearby OpenStreetMap service categories. For each sampled endpoint, I identified the nearest OSM service node within 250 meters and grouped services into broader categories such as transit, food and drink, retail, tourism, office, health, recreation, and education.

The results show that weekend trip endpoints are more likely to fall outside a mapped service node within 250 meters, while weekday endpoints are more strongly associated with transit, food and drink, office, and retail services. Tourism is slightly more weekend-oriented. This suggests that weekday Divvy trips are more connected to dense urban service environments and routine activity areas, while weekend trips are relatively more dispersed and slightly more connected to leisure or tourism-oriented destinations.

This analysis should be interpreted carefully because the OSM enrichment uses nearest mapped service nodes, not complete land-use polygons. Large parks, lakefront areas, and open recreational spaces may be underrepresented if they are not represented as nearby OSM nodes.

In [ ]:
osm_summary = read_output_csv("osm_service_summary_by_day_type.csv")
osm_comparison = read_output_csv("osm_service_weekend_weekday_comparison.csv")

display(osm_summary.sort_values(
    ["day_type", "share_percent"],
    ascending=[True, False]
))

display(osm_comparison.sort_values(
    "weekend_minus_weekday_percent",
    ascending=False
))

In [ ]:
show_png("osm_service_share_by_day_type.png", figsize=(10, 5))

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUTPUTS = ROOT / "outputs"
FIGURES = ROOT / "figures"

comparison = pd.read_csv(OUTPUTS / "osm_service_weekend_weekday_comparison.csv")

label_map = {
    "no_service_within_250m": "No mapped service within 250m",
    "food_drink": "Food/drink",
    "transit": "Transit",
    "office": "Office",
    "retail": "Retail",
    "tourism": "Tourism",
    "recreation": "Recreation",
    "health": "Health",
    "education": "Education",
    "other_service": "Other service",
}

plot_df = comparison.copy()
plot_df["label"] = plot_df["nearest_service_category"].map(label_map).fillna(
    plot_df["nearest_service_category"]
)

plot_df = plot_df.sort_values("weekend_minus_weekday_percent")

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.barh(
    plot_df["label"],
    plot_df["weekend_minus_weekday_percent"],
)

ax.axvline(0, color="black", linewidth=1)

ax.set_title(
    "Weekend minus weekday difference in OSM service environments near trip endpoints"
)
ax.set_xlabel("Weekend share minus weekday share, percentage points")
ax.set_ylabel("Nearest OSM service category")

ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
output_path = FIGURES / "osm_service_weekend_minus_weekday_including_no_service.png"
plt.savefig(output_path, dpi=220, bbox_inches="tight")
plt.close()

print(f"Saved figure to: {output_path}")

In [ ]:
show_png("osm_service_weekend_minus_weekday_including_no_service.png", figsize=(10, 6))

## Station-level Dwell Time Proxy

The dwell proxy is calculated as the time from an arrival event at a station to the next departure event at the same station. This is a **station-level availability or idle-time proxy**, not true bike-level dwell time, because the public Divvy data does not include `bike_id`.

For the proposal, this metric should stay simple. The chart below only compares the median dwell proxy for weekdays and weekends. The main takeaway is that the two medians are very close, so dwell proxy is supporting evidence rather than a main source of weekday/weekend difference.


In [ ]:
dwell_summary_for_display = dwell_summary.copy()
for column in ["mean", "median", "q1", "q3"]:
    dwell_summary_for_display[column] = dwell_summary_for_display[column].round(2)

display(dwell_summary_for_display[["day_type", "count", "median", "mean"]])


def plot_dwell_median_bar(summary: pd.DataFrame, output_path: Path) -> None:
    plot_df = summary.set_index("day_type").loc[["weekday", "weekend"]].reset_index()
    colors = ["#4C78A8", "#F58518"]

    fig, ax = plt.subplots(figsize=(6.2, 4.4))
    bars = ax.bar(plot_df["day_type"], plot_df["median"], color=colors, width=0.58)

    for bar, median in zip(bars, plot_df["median"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.45,
            f"{median:.1f} min",
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold",
        )

    weekday_median = float(plot_df.loc[plot_df["day_type"] == "weekday", "median"].iloc[0])
    weekend_median = float(plot_df.loc[plot_df["day_type"] == "weekend", "median"].iloc[0])
    difference = weekend_median - weekday_median

    ax.set_title("Median Station-level Dwell Proxy by Day Type")
    ax.set_xlabel("Day type based on arrival time")
    ax.set_ylabel("Median dwell proxy (minutes)")
    ax.set_ylim(0, max(plot_df["median"]) * 1.35)
    ax.grid(axis="y", alpha=0.25)
    ax.text(
        0.5,
        0.88,
        f"Weekend median is {abs(difference):.1f} min {'lower' if difference < 0 else 'higher'} than weekday.",
        transform=ax.transAxes,
        ha="center",
        fontsize=10,
        bbox={"facecolor": "white", "alpha": 0.9, "edgecolor": "#cccccc"},
    )
    fig.tight_layout()
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.show()


dwell_bar_path = FIGURES / "dwell_proxy_median_bar.png"
plot_dwell_median_bar(dwell_summary, dwell_bar_path)
print(f"Saved dwell proxy median bar chart to {dwell_bar_path}")


## Recommended Proposal Figures

The strongest figures for the revised proposal and final presentation are:

1. `trip_duration_ccdf.png`, because it shows the long-tailed distribution of trip duration.
2. `trip_distance_ccdf.png`, because it shows the distributional structure of straight-line trip distance.
3. `interactive_start_location_heatmap.html` or `start_location_heatmap_with_basemap.html`, because it adds geographic reference to the spatial heatmap.
4. `osm_service_weekend_minus_weekday_including_no_service.png`, because it directly addresses the OpenStreetMap destination-service analysis.

## Final Project Deliverables

The final project will produce:

1. A reproducible cleaned 2025 Divvy trip dataset generated from the raw monthly files.
2. Summary tables for weekday/weekend trip duration and straight-line distance.
3. Distribution-fit results and CCDF plots for trip duration and distance.
4. Static and interactive spatial heatmaps, including a basemap-supported version.
5. OpenStreetMap destination service summaries comparing weekday and weekend trip endpoints.
6. A station-level dwell time proxy as a secondary activity measure.
7. A final notebook that documents the complete workflow, figures, tables, limitations, and revised findings.

## Limitations and Interpretation Notes

This analysis has several limitations. First, the straight-line trip distance is calculated using the haversine formula between start and end coordinates, so it does not represent the actual bike route distance on the street network.

Second, the OpenStreetMap destination service analysis identifies nearby mapped service nodes within 250 meters of sampled trip endpoints. This does not prove that a rider visited that service. It only describes the type of urban service environment near the trip destination. Large parks, lakefront spaces, plazas, and other open recreational areas may be underrepresented if they are mapped as polygons rather than service nodes.

Third, the station-level dwell time measure is only a proxy. Because the public Divvy dataset does not include bike IDs, true bike-level dwell time cannot be measured. The dwell proxy is therefore interpreted as a station-level activity or availability indicator, not as an individual stop duration.

Finally, the weekday/weekend comparison is descriptive rather than causal. Differences between weekday and weekend trips may reflect commuting, leisure activity, seasonality, weather, special events, tourism, and neighborhood context. The analysis is designed to describe these patterns and provide interpretable evidence, not to claim that day type alone causes the observed differences.

## Recommended Figures for Final Report and Presentation

The most important figures for the final report and pitch are:

1. Trip duration CCDF, because it shows the long-tailed distribution of trip times and directly responds to the distributional baseline feedback.
2. Trip distance CCDF, because it shows the distributional structure of straight-line trip distances.
3. Start-location heatmap with basemap, because it provides geographic reference for interpreting spatial density.
4. OSM service weekend-minus-weekday chart, because it adds behavioral depth by comparing the service environments near trip endpoints.

The OSM service difference chart is especially important because it addresses the professor’s suggestion to use OpenStreetMap service information near end locations.

## Summary of Revised Findings

After incorporating the feedback, the project now goes beyond a basic weekday/weekend comparison. The revised analysis shows that weekend Divvy trips have longer typical durations and slightly longer straight-line distances than weekday trips. The CCDF plots also show that weekend trips have a heavier upper tail, especially for trip duration, meaning that longer trips are more common on weekends.

The spatial analysis was also improved by adding a basemap to the heatmap. This makes the spatial patterns easier to interpret in relation to downtown Chicago, the lakefront, parks, streets, and surrounding neighborhoods.

The OpenStreetMap destination service analysis adds the most important new behavioral depth. Weekday trip endpoints are more strongly associated with dense service environments such as food and drink, transit, office, and retail. Weekend endpoints are more likely to fall outside a mapped service node within 250 meters and are slightly more associated with tourism. This suggests that weekday trips are more connected to routine urban activity areas, while weekend trips are more dispersed and somewhat more connected to leisure or tourism-oriented destinations.

Overall, the revised project now addresses trip distributions, spatial context, and destination service environments, rather than relying only on median trip statistics and raw spatial density.


## Analysis Plan and Paper Connection

The final analysis follows a revised six-step pipeline.

First, I clean and validate the full-year 2025 Divvy dataset, keeping a reproducible record of filtering decisions and final row counts. This step ensures that timestamps, coordinates, trip duration, and straight-line distance are usable for analysis.

Second, I compare weekday and weekend trip duration and straight-line trip length. Instead of relying only on medians, I examine the full distributions using log-scaled histograms and complementary cumulative distribution functions (CCDFs). I also report descriptive lognormal and power-law tail parameters to better represent the right-skewed and long-tailed structure of the mobility data.

Third, I analyze spatial start-location density using both static heatmaps and an interactive heatmap with an OpenStreetMap/Carto basemap. The basemap provides geographic reference for interpreting hotspots in relation to the lakefront, downtown Chicago, parks, major corridors, and surrounding neighborhoods.

Fourth, I compare normalized weekday and weekend start-location density. Because weekday trips are more common than weekend trips, normalized density is used instead of raw counts. This helps identify places that are relatively more weekday-oriented or weekend-oriented.

Fifth, I enrich sampled trip end locations with nearby OpenStreetMap service categories. For each sampled endpoint, I identify the nearest mapped OSM service node within 250 meters and classify it into broader categories such as transit, food and drink, retail, tourism, office, health, recreation, and education. This adds behavioral depth by comparing not only where trips occur, but also what types of urban services are accessed near weekday and weekend destinations.

Sixth, I include the station-level dwell time proxy as a secondary measure of station activity. Because the public Divvy data does not include bike IDs, this is not a true bike-level dwell measure. It is used only as a station-level availability proxy.

This revised analysis connects directly to Zhou (2015), who analyzed Chicago Divvy spatiotemporal biking behavior, including weekday and weekend differences. It also connects to Chiariotti et al. (2018), where bike-share station availability and rebalancing are central concerns. Compared with these studies, this project focuses on a reproducible 2025 weekday/weekend comparison with distributional analysis, basemap-supported spatial interpretation, normalized heatmaps, and OpenStreetMap-based destination service analysis.

---

# Final Project Extension Pipeline

The following sections merge the code from `openmap.ipynb`, `statistics.ipynb`, and `validation_with_weather.ipynb` into this proposal summary notebook. Run the notebook from top to bottom after placing the required input files in the expected folders/paths.


---

# OpenMap / End-Station + OSM + Weather Map Pipeline

Source notebook merged from: `openmap.ipynb`


In [ ]:
%pip install geopy openpyxl

In [ ]:
import os
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# =========================
# 1. File paths
# =========================
STATION_COUNTS_PATH = "station_name_counts.xlsx"

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(OUTPUT_DIR, "top20_end_stations_geocoded.csv")

# =========================
# 2. Read top 20 end stations
# =========================
end_counts = pd.read_excel(
    STATION_COUNTS_PATH,
    sheet_name="End Station Counts"
)

end_counts = end_counts.rename(columns={
    "station": "end_station_name",
    "counts": "end_trip_count"
})

end_counts["end_station_name"] = (
    end_counts["end_station_name"]
    .astype(str)
    .str.strip()
)

top20_end_counts = (
    end_counts
    .sort_values("end_trip_count", ascending=False)
    .head(20)
    .copy()
)

# =========================
# 3. Geocode station names
# =========================
geolocator = Nominatim(user_agent="divvy_top20_station_geocoder")

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.2,
    max_retries=2,
    error_wait_seconds=5
)

latitudes = []
longitudes = []
matched_addresses = []

for station in top20_end_counts["end_station_name"]:
    query = f"{station}, Chicago, Illinois, USA"
    print("Geocoding:", query)

    location = geocode(query)

    if location is None:
        latitudes.append(None)
        longitudes.append(None)
        matched_addresses.append(None)
    else:
        latitudes.append(location.latitude)
        longitudes.append(location.longitude)
        matched_addresses.append(location.address)

top20_end_counts["end_lat"] = latitudes
top20_end_counts["end_lng"] = longitudes
top20_end_counts["matched_address"] = matched_addresses

# =========================
# 4. Save CSV output
# =========================
top20_end_counts.to_csv(OUTPUT_CSV, index=False)

print("\nSaved CSV to:")
print(OUTPUT_CSV)

display(top20_end_counts)

In [ ]:
# Read top 20 geocoded end station table
end_stations = pd.read_csv("top20_end_stations_geocoded.csv")

# Check columns and preview
print(end_stations.shape)
print(end_stations.columns)
display(end_stations.head())

# Keep only rows with valid coordinates
end_stations = end_stations.dropna(subset=["end_lat", "end_lng", "end_trip_count"]).copy()

# Optional: keep only coordinates inside Chicago area
end_stations = end_stations[
    end_stations["end_lat"].between(41.5, 42.2) &
    end_stations["end_lng"].between(-88.1, -87.3)
].copy()

display(end_stations)

In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap


# Chicago center
center_lat = end_stations["end_lat"].mean()
center_lng = end_stations["end_lng"].mean()

m = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=12,
    tiles="OpenStreetMap"
)

for _, row in end_stations.iterrows():
    radius = 4 + np.log1p(row["end_trip_count"]) / 1.5
    
    folium.CircleMarker(
        location=[row["end_lat"], row["end_lng"]],
        radius=radius,
        popup=(
            f"<b>{row['end_station_name']}</b><br>"
            f"End trips: {row['end_trip_count']}<br>"
            f"Lat: {row['end_lat']}<br>"
            f"Lng: {row['end_lng']}"
        ),
        tooltip=f"{row['end_station_name']} ({row['end_trip_count']})",
        fill=True,
        fill_opacity=0.65
    ).add_to(m)

m.save("top20_end_station_points_map.html")
m

In [ ]:
m_heat = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=12,
    tiles="OpenStreetMap"
)

heat_data = end_stations[
    ["end_lat", "end_lng", "end_trip_count"]
].values.tolist()

HeatMap(
    heat_data,
    radius=25,
    blur=18,
    max_zoom=13
).add_to(m_heat)

m_heat.save("top20_end_station_weighted_heatmap.html")
m_heat

In [ ]:
m_combined = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=12,
    tiles="OpenStreetMap"
)

# Layer 1: weighted heatmap
heat_layer = folium.FeatureGroup(name="Weighted heatmap: Top 20 end stations")

heat_data = end_stations[
    ["end_lat", "end_lng", "end_trip_count"]
].values.tolist()

HeatMap(
    heat_data,
    radius=25,
    blur=18,
    max_zoom=13
).add_to(heat_layer)

heat_layer.add_to(m_combined)

# Layer 2: station point markers
point_layer = folium.FeatureGroup(name="Point markers: Top 20 end stations")

for _, row in end_stations.iterrows():
    radius = 4 + np.log1p(row["end_trip_count"]) / 1.5
    
    folium.CircleMarker(
        location=[row["end_lat"], row["end_lng"]],
        radius=radius,
        popup=(
            f"<b>{row['end_station_name']}</b><br>"
            f"End trips: {row['end_trip_count']}<br>"
            f"Matched address: {row.get('matched_address', '')}"
        ),
        tooltip=f"{row['end_station_name']} ({row['end_trip_count']})",
        fill=True,
        fill_opacity=0.65
    ).add_to(point_layer)

point_layer.add_to(m_combined)

folium.LayerControl().add_to(m_combined)

m_combined.save("top20_end_station_combined_map.html")
m_combined

In [ ]:
%pip install osmnx geopandas folium openpyxl

In [ ]:
import os
import time
import pandas as pd
import numpy as np

import osmnx as ox
import geopandas as gpd
import folium
from folium.plugins import HeatMap

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

STATION_CSV = "top20_end_stations_geocoded.csv"

stations = pd.read_csv(STATION_CSV)

stations = stations.dropna(subset=["end_lat", "end_lng"]).copy()

stations = stations[
    stations["end_lat"].between(41.5, 42.2) &
    stations["end_lng"].between(-88.1, -87.3)
].copy()

display(stations)

In [ ]:
OSM_TAGS = {
    "amenity": True,
    "shop": True,
    "tourism": True,
    "office": True,
    "leisure": True,
    "public_transport": True,
    "railway": ["station", "subway_entrance", "tram_stop"],
    "healthcare": True
}

In [ ]:
def get_tag(row, col):
    if col not in row.index:
        return ""
    value = row[col]
    if pd.isna(value):
        return ""
    return str(value).lower()


def classify_osm_service(row):
    amenity = get_tag(row, "amenity")
    shop = get_tag(row, "shop")
    tourism = get_tag(row, "tourism")
    office = get_tag(row, "office")
    leisure = get_tag(row, "leisure")
    public_transport = get_tag(row, "public_transport")
    railway = get_tag(row, "railway")
    healthcare = get_tag(row, "healthcare")

    food_drink = {
        "restaurant", "cafe", "bar", "pub", "fast_food",
        "food_court", "ice_cream", "biergarten"
    }

    transit = {
        "bus_station", "ferry_terminal", "taxi"
    }

    education = {
        "school", "college", "university", "library",
        "kindergarten"
    }

    health = {
        "hospital", "clinic", "doctors", "dentist", "pharmacy"
    }

    if amenity in food_drink:
        return "food_drink"

    if amenity in transit or public_transport != "" or railway in ["station", "subway_entrance", "tram_stop"]:
        return "transit"

    if shop != "":
        return "retail"

    if tourism != "":
        return "tourism"

    if office != "":
        return "office"

    if amenity in health or healthcare != "":
        return "health"

    if leisure != "":
        return "recreation"

    if amenity in education:
        return "education"

    if amenity != "":
        return "other_amenity"

    return "other_service"

In [ ]:
RADIUS_M = 250

all_services = []

for i, row in stations.iterrows():
    station_name = row["end_station_name"]
    lat = row["end_lat"]
    lng = row["end_lng"]
    
    print(f"Querying OSM services around: {station_name}")

    try:
        # OSMnx expects center_point as (lat, lon)
        gdf = ox.features_from_point(
            center_point=(lat, lng),
            tags=OSM_TAGS,
            dist=RADIUS_M
        )

        if gdf.empty:
            print("  No OSM services found.")
            continue

        gdf = gdf.reset_index()
        gdf["end_station_name"] = station_name
        gdf["station_end_trip_count"] = row["end_trip_count"]
        gdf["station_lat"] = lat
        gdf["station_lng"] = lng
        gdf["search_radius_m"] = RADIUS_M

        gdf["service_category"] = gdf.apply(classify_osm_service, axis=1)

        all_services.append(gdf)

        print(f"  Found {len(gdf)} OSM features.")

        # polite pause for Overpass / OSM server
        time.sleep(1)

    except Exception as e:
        print(f"  Error for {station_name}: {e}")

if len(all_services) > 0:
    services_gdf = pd.concat(all_services, ignore_index=True)
else:
    services_gdf = gpd.GeoDataFrame()

print("Total OSM services found:", len(services_gdf))
display(services_gdf.head())

In [ ]:
if len(services_gdf) > 0:
    services_gdf = gpd.GeoDataFrame(services_gdf, geometry="geometry", crs="EPSG:4326")

    # Convert geometries to centroid points for CSV/map display
    services_proj = services_gdf.to_crs("EPSG:26916")
    service_centroids_proj = services_proj.copy()
    service_centroids_proj["geometry"] = services_proj.geometry.centroid
    service_centroids = service_centroids_proj.to_crs("EPSG:4326")

    service_centroids["service_lat"] = service_centroids.geometry.y
    service_centroids["service_lng"] = service_centroids.geometry.x

    useful_cols = [
        "end_station_name",
        "station_end_trip_count",
        "station_lat",
        "station_lng",
        "search_radius_m",
        "service_category",
        "service_lat",
        "service_lng",
        "name",
        "amenity",
        "shop",
        "tourism",
        "office",
        "leisure",
        "public_transport",
        "railway",
        "healthcare"
    ]

    existing_cols = [c for c in useful_cols if c in service_centroids.columns]

    services_output = service_centroids[existing_cols].copy()

    services_output.to_csv(
        os.path.join(OUTPUT_DIR, "top20_end_station_osm_services_250m.csv"),
        index=False
    )

    display(services_output.head())

    print("Saved service detail CSV:")
    print(os.path.join(OUTPUT_DIR, "top20_end_station_osm_services_250m.csv"))

In [ ]:
service_summary = (
    services_output
    .groupby(["end_station_name", "service_category"])
    .size()
    .reset_index(name="service_count")
)

service_summary_wide = (
    service_summary
    .pivot_table(
        index="end_station_name",
        columns="service_category",
        values="service_count",
        fill_value=0
    )
    .reset_index()
)

# Add station trip count back
station_info = stations[["end_station_name", "end_trip_count", "end_lat", "end_lng"]].copy()

service_summary_wide = station_info.merge(
    service_summary_wide,
    on="end_station_name",
    how="left"
).fillna(0)

# Total number of nearby OSM services
category_cols = [
    c for c in service_summary_wide.columns
    if c not in ["end_station_name", "end_trip_count", "end_lat", "end_lng"]
]

service_summary_wide["total_services_250m"] = service_summary_wide[category_cols].sum(axis=1)

service_summary_wide.to_csv(
    os.path.join(OUTPUT_DIR, "top20_end_station_osm_service_counts_250m.csv"),
    index=False
)

display(service_summary_wide)

print("Saved service count CSV:")
print(os.path.join(OUTPUT_DIR, "top20_end_station_osm_service_counts_250m.csv"))

In [ ]:
center_lat = stations["end_lat"].mean()
center_lng = stations["end_lng"].mean()

m = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=13,
    tiles="OpenStreetMap"
)

# Add station markers + 250m search circles
for _, row in stations.iterrows():
    folium.Circle(
        location=[row["end_lat"], row["end_lng"]],
        radius=RADIUS_M,
        fill=False,
        tooltip=f"250m radius: {row['end_station_name']}"
    ).add_to(m)

    folium.CircleMarker(
        location=[row["end_lat"], row["end_lng"]],
        radius=6 + np.log1p(row["end_trip_count"]) / 2,
        popup=(
            f"<b>End station:</b> {row['end_station_name']}<br>"
            f"<b>End trips:</b> {row['end_trip_count']}"
        ),
        fill=True,
        fill_opacity=0.8
    ).add_to(m)

# Add OSM service points
for _, row in services_output.iterrows():
    service_name = row["name"] if "name" in services_output.columns and pd.notna(row.get("name", None)) else "Unnamed service"

    folium.CircleMarker(
        location=[row["service_lat"], row["service_lng"]],
        radius=3,
        popup=(
            f"<b>{service_name}</b><br>"
            f"Category: {row['service_category']}<br>"
            f"Near station: {row['end_station_name']}"
        ),
        fill=True,
        fill_opacity=0.6
    ).add_to(m)

m.save(os.path.join(OUTPUT_DIR, "top20_end_station_osm_services_map.html"))

m

In [ ]:
m_service_heat = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=13,
    tiles="OpenStreetMap"
)

service_heat_data = services_output[
    ["service_lat", "service_lng"]
].dropna().values.tolist()

HeatMap(
    service_heat_data,
    radius=18,
    blur=15,
    max_zoom=13
).add_to(m_service_heat)

m_service_heat.save(os.path.join(OUTPUT_DIR, "top20_end_station_osm_service_heatmap.html"))

m_service_heat

In [ ]:
import os
import pandas as pd
import numpy as np
import folium

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# If variables already exist, you do not need to read them again.
# But this makes the cell runnable by itself.
stations = pd.read_csv("top20_end_stations_geocoded.csv")
services_output = pd.read_csv(os.path.join(OUTPUT_DIR, "top20_end_station_osm_services_250m.csv"))

# Clean coordinates
stations = stations.dropna(subset=["end_lat", "end_lng"]).copy()
services_output = services_output.dropna(subset=["service_lat", "service_lng"]).copy()

# Map center
center_lat = stations["end_lat"].mean()
center_lng = stations["end_lng"].mean()

m = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=13,
    tiles="OpenStreetMap"
)

# =========================
# 1. Station layer
# =========================
station_layer = folium.FeatureGroup(
    name="Top 20 Divvy end stations",
    show=True
)

RADIUS_M = 250

for _, station in stations.iterrows():
    station_name = station["end_station_name"]

    # services near this station
    nearby = services_output[
        services_output["end_station_name"] == station_name
    ].copy()

    # Build popup service list
    if len(nearby) > 0:
        service_items = ""

        for category, group in nearby.groupby("service_category"):
            service_items += f"<b>{category}</b><ul>"

            for _, service in group.head(15).iterrows():
                service_name = service["name"] if "name" in service.index and pd.notna(service["name"]) else "Unnamed service"
                service_items += f"<li>{service_name}</li>"

            if len(group) > 15:
                service_items += f"<li>... and {len(group) - 15} more</li>"

            service_items += "</ul>"
    else:
        service_items = "No mapped services found within 250m."

    popup_html = f"""
    <div style="width: 320px;">
        <h4>{station_name}</h4>
        <p><b>End trips:</b> {station["end_trip_count"]}</p>
        <p><b>Search radius:</b> {RADIUS_M} m</p>
        <hr>
        <p><b>Nearby OSM services:</b></p>
        {service_items}
    </div>
    """

    # Radius circle
    folium.Circle(
        location=[station["end_lat"], station["end_lng"]],
        radius=RADIUS_M,
        fill=False,
        tooltip=f"250m radius around {station_name}"
    ).add_to(station_layer)

    # Station marker
    folium.CircleMarker(
        location=[station["end_lat"], station["end_lng"]],
        radius=6 + np.log1p(station["end_trip_count"]) / 2,
        popup=folium.Popup(popup_html, max_width=360),
        tooltip=f"{station_name} ({station['end_trip_count']} end trips)",
        fill=True,
        fill_opacity=0.85
    ).add_to(station_layer)

station_layer.add_to(m)

# =========================
# 2. Service category checkbox layers
# =========================
service_categories = sorted(services_output["service_category"].dropna().unique())

# Optional icon/color settings
category_colors = {
    "food_drink": "red",
    "retail": "blue",
    "transit": "green",
    "tourism": "purple",
    "office": "gray",
    "health": "pink",
    "recreation": "orange",
    "education": "cadetblue",
    "other_amenity": "lightgray",
    "other_service": "lightgray"
}

for category in service_categories:
    category_layer = folium.FeatureGroup(
        name=f"Service: {category}",
        show=False
    )

    temp = services_output[
        services_output["service_category"] == category
    ].copy()

    for _, service in temp.iterrows():
        service_name = service["name"] if "name" in service.index and pd.notna(service["name"]) else "Unnamed service"

        popup_html = f"""
        <div style="width: 260px;">
            <b>{service_name}</b><br>
            <b>Category:</b> {category}<br>
            <b>Near station:</b> {service["end_station_name"]}<br>
            <b>Amenity:</b> {service.get("amenity", "")}<br>
            <b>Shop:</b> {service.get("shop", "")}<br>
            <b>Tourism:</b> {service.get("tourism", "")}<br>
            <b>Office:</b> {service.get("office", "")}<br>
            <b>Leisure:</b> {service.get("leisure", "")}
        </div>
        """

        folium.CircleMarker(
            location=[service["service_lat"], service["service_lng"]],
            radius=4,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"{category}: {service_name}",
            fill=True,
            fill_opacity=0.75,
            color=category_colors.get(category, "black"),
            fill_color=category_colors.get(category, "black")
        ).add_to(category_layer)

    category_layer.add_to(m)

# =========================
# 3. Add layer checkbox control
# =========================
folium.LayerControl(collapsed=False).add_to(m)

output_map = os.path.join(OUTPUT_DIR, "top20_end_station_services_checkbox_map.html")
m.save(output_map)

print("Saved map to:")
print(output_map)

m

In [ ]:
import os
import pandas as pd
import numpy as np

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


CLEAN_DIVVY_PATH = "divvy_2025_cleaned.csv"

usecols = [
    "end_station_name",
    "end_lat",
    "end_lng",
    "day_type"
]

chunk_size = 500_000

count_parts = []
coord_parts = []

for chunk in pd.read_csv(CLEAN_DIVVY_PATH, usecols=usecols, chunksize=chunk_size):
    # clean basic fields
    chunk = chunk.dropna(subset=["end_station_name", "end_lat", "end_lng", "day_type"]).copy()
    
    chunk["end_station_name"] = chunk["end_station_name"].astype(str).str.strip()
    chunk["day_type"] = chunk["day_type"].astype(str).str.lower().str.strip()
    
    chunk["end_lat"] = pd.to_numeric(chunk["end_lat"], errors="coerce")
    chunk["end_lng"] = pd.to_numeric(chunk["end_lng"], errors="coerce")
    
    chunk = chunk.dropna(subset=["end_lat", "end_lng"]).copy()
    
    # keep only weekday/weekend
    chunk = chunk[chunk["day_type"].isin(["weekday", "weekend"])].copy()
    
    # Chicago coordinate sanity filter
    chunk = chunk[
        chunk["end_lat"].between(41.5, 42.2) &
        chunk["end_lng"].between(-88.1, -87.3)
    ].copy()
    
    # Count return stations by day type
    counts = (
        chunk
        .groupby(["day_type", "end_station_name"])
        .size()
        .reset_index(name="return_count")
    )
    count_parts.append(counts)
    
    # Coordinate aggregation
    coords = (
        chunk
        .groupby("end_station_name", as_index=False)
        .agg(
            lat_sum=("end_lat", "sum"),
            lng_sum=("end_lng", "sum"),
            coord_n=("end_lat", "count")
        )
    )
    coord_parts.append(coords)

# Combine counts across chunks
station_day_counts = (
    pd.concat(count_parts, ignore_index=True)
    .groupby(["day_type", "end_station_name"], as_index=False)
    .agg(return_count=("return_count", "sum"))
)

# Combine coordinate sums across chunks
station_coords = (
    pd.concat(coord_parts, ignore_index=True)
    .groupby("end_station_name", as_index=False)
    .agg(
        lat_sum=("lat_sum", "sum"),
        lng_sum=("lng_sum", "sum"),
        coord_n=("coord_n", "sum")
    )
)

station_coords["end_lat"] = station_coords["lat_sum"] / station_coords["coord_n"]
station_coords["end_lng"] = station_coords["lng_sum"] / station_coords["coord_n"]

station_coords = station_coords[["end_station_name", "end_lat", "end_lng"]]

print("Station-day counts:")
display(station_day_counts.head())

print("Station coordinates:")
display(station_coords.head())

In [ ]:
# Total returns by day type
station_day_counts["total_returns_by_day_type"] = (
    station_day_counts
    .groupby("day_type")["return_count"]
    .transform("sum")
)

station_day_counts["return_share"] = (
    station_day_counts["return_count"] /
    station_day_counts["total_returns_by_day_type"]
)

# Pivot to wide format
station_compare = station_day_counts.pivot_table(
    index="end_station_name",
    columns="day_type",
    values=["return_count", "return_share"],
    fill_value=0
).reset_index()

# Flatten column names
station_compare.columns = [
    "_".join([str(x) for x in col if x != ""])
    for col in station_compare.columns
]

# Ensure columns exist
for col in [
    "return_count_weekday",
    "return_count_weekend",
    "return_share_weekday",
    "return_share_weekend"
]:
    if col not in station_compare.columns:
        station_compare[col] = 0

# Merge coordinates
station_compare = station_compare.merge(
    station_coords,
    on="end_station_name",
    how="left"
)

# Calculate differences
station_compare["total_return_count"] = (
    station_compare["return_count_weekday"] +
    station_compare["return_count_weekend"]
)

station_compare["weekend_minus_weekday_share"] = (
    station_compare["return_share_weekend"] -
    station_compare["return_share_weekday"]
)

station_compare["weekday_minus_weekend_share"] = (
    station_compare["return_share_weekday"] -
    station_compare["return_share_weekend"]
)

station_compare["weekend_weekday_share_ratio"] = (
    (station_compare["return_share_weekend"] + 1e-9) /
    (station_compare["return_share_weekday"] + 1e-9)
)

station_compare["orientation"] = np.where(
    station_compare["weekend_minus_weekday_share"] > 0,
    "weekend_oriented",
    "weekday_oriented"
)

# Remove very tiny stations to avoid noise
MIN_TOTAL_RETURNS = 50

station_compare_filtered = station_compare[
    station_compare["total_return_count"] >= MIN_TOTAL_RETURNS
].copy()

# Save full comparison table
output_csv = os.path.join(
    OUTPUT_DIR,
    "layer1_weekday_weekend_return_station_comparison.csv"
)

station_compare_filtered.to_csv(output_csv, index=False)

print("Saved:", output_csv)
print(station_compare_filtered.shape)

display(
    station_compare_filtered
    .sort_values("weekend_minus_weekday_share", ascending=False)
    .head(20)
)

In [ ]:
# Top stations by raw weekday return count
top20_weekday_returns = (
    station_compare_filtered
    .sort_values("return_count_weekday", ascending=False)
    .head(20)
)

# Top stations by raw weekend return count
top20_weekend_returns = (
    station_compare_filtered
    .sort_values("return_count_weekend", ascending=False)
    .head(20)
)

# Stations relatively more weekend-oriented
top20_weekend_oriented = (
    station_compare_filtered
    .sort_values("weekend_minus_weekday_share", ascending=False)
    .head(20)
)

# Stations relatively more weekday-oriented
top20_weekday_oriented = (
    station_compare_filtered
    .sort_values("weekend_minus_weekday_share", ascending=True)
    .head(20)
)

# Save outputs
top20_weekday_returns.to_csv(
    os.path.join(OUTPUT_DIR, "top20_weekday_return_stations.csv"),
    index=False
)

top20_weekend_returns.to_csv(
    os.path.join(OUTPUT_DIR, "top20_weekend_return_stations.csv"),
    index=False
)

top20_weekend_oriented.to_csv(
    os.path.join(OUTPUT_DIR, "top20_weekend_oriented_return_stations.csv"),
    index=False
)

top20_weekday_oriented.to_csv(
    os.path.join(OUTPUT_DIR, "top20_weekday_oriented_return_stations.csv"),
    index=False
)

print("Top 20 weekday return stations")
display(top20_weekday_returns[[
    "end_station_name",
    "return_count_weekday",
    "return_share_weekday",
    "end_lat",
    "end_lng"
]])

print("Top 20 weekend return stations")
display(top20_weekend_returns[[
    "end_station_name",
    "return_count_weekend",
    "return_share_weekend",
    "end_lat",
    "end_lng"
]])

print("Top 20 weekend-oriented stations")
display(top20_weekend_oriented[[
    "end_station_name",
    "return_count_weekday",
    "return_count_weekend",
    "return_share_weekday",
    "return_share_weekend",
    "weekend_minus_weekday_share",
    "weekend_weekday_share_ratio",
    "end_lat",
    "end_lng"
]])

print("Top 20 weekday-oriented stations")
display(top20_weekday_oriented[[
    "end_station_name",
    "return_count_weekday",
    "return_count_weekend",
    "return_share_weekday",
    "return_share_weekend",
    "weekend_minus_weekday_share",
    "weekend_weekday_share_ratio",
    "end_lat",
    "end_lng"
]])

In [ ]:
import folium
from folium.plugins import HeatMap

map_df = station_compare_filtered.dropna(subset=["end_lat", "end_lng"]).copy()

center_lat = map_df["end_lat"].mean()
center_lng = map_df["end_lng"].mean()

m_heat = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=12,
    tiles="OpenStreetMap"
)

# Weekday weighted heatmap
weekday_layer = folium.FeatureGroup(
    name="Weekday return station heatmap",
    show=True
)

weekday_heat_data = map_df[
    ["end_lat", "end_lng", "return_count_weekday"]
].dropna().values.tolist()

HeatMap(
    weekday_heat_data,
    radius=18,
    blur=15,
    max_zoom=13
).add_to(weekday_layer)

weekday_layer.add_to(m_heat)

# Weekend weighted heatmap
weekend_layer = folium.FeatureGroup(
    name="Weekend return station heatmap",
    show=False
)

weekend_heat_data = map_df[
    ["end_lat", "end_lng", "return_count_weekend"]
].dropna().values.tolist()

HeatMap(
    weekend_heat_data,
    radius=18,
    blur=15,
    max_zoom=13
).add_to(weekend_layer)

weekend_layer.add_to(m_heat)

folium.LayerControl(collapsed=False).add_to(m_heat)

output_map = os.path.join(
    OUTPUT_DIR,
    "layer1_weekday_weekend_return_station_heatmap.html"
)

m_heat.save(output_map)

print("Saved:", output_map)
m_heat

In [ ]:
m_diff = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=12,
    tiles="OpenStreetMap"
)

weekend_layer = folium.FeatureGroup(
    name="Weekend-oriented return stations",
    show=True
)

weekday_layer = folium.FeatureGroup(
    name="Weekday-oriented return stations",
    show=True
)

diff_stations = pd.concat([
    top20_weekend_oriented,
    top20_weekday_oriented
], ignore_index=True)

for _, row in diff_stations.iterrows():
    popup_html = f"""
    <div style="width: 330px;">
        <h4>{row['end_station_name']}</h4>
        <b>Weekday returns:</b> {int(row['return_count_weekday'])}<br>
        <b>Weekend returns:</b> {int(row['return_count_weekend'])}<br>
        <b>Weekday share:</b> {row['return_share_weekday']:.6f}<br>
        <b>Weekend share:</b> {row['return_share_weekend']:.6f}<br>
        <b>Weekend - weekday share:</b> {row['weekend_minus_weekday_share']:.6f}<br>
        <b>Weekend / weekday share ratio:</b> {row['weekend_weekday_share_ratio']:.2f}<br>
        <b>Orientation:</b> {row['orientation']}
    </div>
    """

    radius = 6 + np.log1p(abs(row["weekend_minus_weekday_share"]) * 1_000_000)

    marker = folium.CircleMarker(
        location=[row["end_lat"], row["end_lng"]],
        radius=radius,
        popup=folium.Popup(popup_html, max_width=360),
        tooltip=f"{row['orientation']}: {row['end_station_name']}",
        fill=True,
        fill_opacity=0.75
    )

    if row["weekend_minus_weekday_share"] > 0:
        marker.add_to(weekend_layer)
    else:
        marker.add_to(weekday_layer)

weekend_layer.add_to(m_diff)
weekday_layer.add_to(m_diff)

folium.LayerControl(collapsed=False).add_to(m_diff)

output_map = os.path.join(
    OUTPUT_DIR,
    "layer1_weekday_weekend_oriented_station_map.html"
)

m_diff.save(output_map)

print("Saved:", output_map)
m_diff

In [ ]:
import os
import pandas as pd
import numpy as np

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

WEATHER_PATH = "daily_trip_weather_summary.csv"

weather = pd.read_csv(WEATHER_PATH)

# Parse date
weather["date"] = pd.to_datetime(weather["date"], errors="coerce").dt.date

# Average temperature
weather["TAVG"] = (weather["TMAX"] + weather["TMIN"]) / 2

# Weather labels
# NOAA style data usually uses Fahrenheit for temperature and inches for precipitation
weather["is_rainy"] = weather["PRCP"].fillna(0) > 0
weather["is_cold"] = weather["TAVG"] <= 40

# Snow can be represented by snowfall or snow depth
weather["is_snowy"] = (
    weather["SNOW"].fillna(0) > 0
) | (
    weather["SNWD"].fillna(0) > 0
)

weather["rain_condition"] = np.where(weather["is_rainy"], "rainy", "dry")
weather["cold_condition"] = np.where(weather["is_cold"], "cold", "mild")
weather["snow_condition"] = np.where(weather["is_snowy"], "snowy", "no_snow")

weather_labels = weather[[
    "date",
    "TMAX",
    "TMIN",
    "TAVG",
    "PRCP",
    "SNOW",
    "SNWD",
    "rain_condition",
    "cold_condition",
    "snow_condition"
]].copy()

weather_labels.to_csv(
    os.path.join(OUTPUT_DIR, "weather_labels_2025.csv"),
    index=False
)

display(weather_labels.head())
print(weather_labels[["rain_condition", "cold_condition", "snow_condition"]].value_counts())

In [ ]:
CLEAN_DIVVY_PATH = "divvy_2025_cleaned.csv"

usecols = [
    "date",
    "day_type",
    "end_station_name",
    "end_lat",
    "end_lng"
]

chunk_size = 500_000

count_parts = []
coord_parts = []

for chunk in pd.read_csv(CLEAN_DIVVY_PATH, usecols=usecols, chunksize=chunk_size):
    # Basic cleaning
    chunk = chunk.dropna(subset=["date", "day_type", "end_station_name", "end_lat", "end_lng"]).copy()
    
    chunk["date"] = pd.to_datetime(chunk["date"], errors="coerce").dt.date
    chunk["day_type"] = chunk["day_type"].astype(str).str.lower().str.strip()
    chunk["end_station_name"] = chunk["end_station_name"].astype(str).str.strip()
    
    chunk["end_lat"] = pd.to_numeric(chunk["end_lat"], errors="coerce")
    chunk["end_lng"] = pd.to_numeric(chunk["end_lng"], errors="coerce")
    
    chunk = chunk.dropna(subset=["date", "end_lat", "end_lng"]).copy()
    
    # Keep only weekday/weekend
    chunk = chunk[chunk["day_type"].isin(["weekday", "weekend"])].copy()
    
    # Chicago coordinate sanity filter
    chunk = chunk[
        chunk["end_lat"].between(41.5, 42.2) &
        chunk["end_lng"].between(-88.1, -87.3)
    ].copy()
    
    # Merge weather labels by date
    chunk = chunk.merge(
        weather_labels[[
            "date",
            "rain_condition",
            "cold_condition",
            "snow_condition",
            "TAVG",
            "PRCP",
            "SNOW",
            "SNWD"
        ]],
        on="date",
        how="left"
    )
    
    chunk = chunk.dropna(subset=["rain_condition", "cold_condition", "snow_condition"]).copy()
    
    # Create station counts for each weather condition type
    condition_info = {
        "rain": "rain_condition",
        "cold": "cold_condition",
        "snow": "snow_condition"
    }
    
    for condition_type, condition_col in condition_info.items():
        counts = (
            chunk
            .groupby(["day_type", condition_col, "end_station_name"])
            .size()
            .reset_index(name="return_count")
            .rename(columns={condition_col: "condition_value"})
        )
        
        counts["condition_type"] = condition_type
        count_parts.append(counts)
    
    # Coordinates
    coords = (
        chunk
        .groupby("end_station_name", as_index=False)
        .agg(
            lat_sum=("end_lat", "sum"),
            lng_sum=("end_lng", "sum"),
            coord_n=("end_lat", "count")
        )
    )
    coord_parts.append(coords)

# Combine station-weather counts
station_weather_counts = (
    pd.concat(count_parts, ignore_index=True)
    .groupby(
        ["condition_type", "day_type", "condition_value", "end_station_name"],
        as_index=False
    )
    .agg(return_count=("return_count", "sum"))
)

# Combine coordinates
station_coords = (
    pd.concat(coord_parts, ignore_index=True)
    .groupby("end_station_name", as_index=False)
    .agg(
        lat_sum=("lat_sum", "sum"),
        lng_sum=("lng_sum", "sum"),
        coord_n=("coord_n", "sum")
    )
)

station_coords["end_lat"] = station_coords["lat_sum"] / station_coords["coord_n"]
station_coords["end_lng"] = station_coords["lng_sum"] / station_coords["coord_n"]

station_coords = station_coords[["end_station_name", "end_lat", "end_lng"]]

station_weather_counts.to_csv(
    os.path.join(OUTPUT_DIR, "station_weather_return_counts.csv"),
    index=False
)

station_coords.to_csv(
    os.path.join(OUTPUT_DIR, "station_coordinates_from_cleaned_divvy.csv"),
    index=False
)

display(station_weather_counts.head())
display(station_coords.head())

In [ ]:
def weather_station_difference(
    station_weather_counts,
    station_coords,
    condition_type,
    positive_value,
    baseline_value,
    day_type_filter=None,
    min_total_returns=50,
    output_name="weather_station_difference.csv"
):
    """
    Compare station return shares under two weather conditions.
    
    Example:
    condition_type = "rain"
    positive_value = "rainy"
    baseline_value = "dry"
    
    Difference = positive condition share - baseline condition share
    """
    
    temp = station_weather_counts[
        station_weather_counts["condition_type"] == condition_type
    ].copy()
    
    if day_type_filter is not None:
        temp = temp[temp["day_type"] == day_type_filter].copy()
    
    # Sum across day types if no day_type_filter
    temp = (
        temp
        .groupby(["condition_value", "end_station_name"], as_index=False)
        .agg(return_count=("return_count", "sum"))
    )
    
    # Keep only the two target conditions
    temp = temp[temp["condition_value"].isin([positive_value, baseline_value])].copy()
    
    # Return share within each condition
    temp["total_returns_by_condition"] = (
        temp
        .groupby("condition_value")["return_count"]
        .transform("sum")
    )
    
    temp["return_share"] = (
        temp["return_count"] /
        temp["total_returns_by_condition"]
    )
    
    # Pivot wide
    wide = temp.pivot_table(
        index="end_station_name",
        columns="condition_value",
        values=["return_count", "return_share"],
        fill_value=0
    ).reset_index()
    
    wide.columns = [
        "_".join([str(x) for x in col if x != ""])
        for col in wide.columns
    ]
    
    # Ensure columns exist
    for col in [
        f"return_count_{positive_value}",
        f"return_count_{baseline_value}",
        f"return_share_{positive_value}",
        f"return_share_{baseline_value}"
    ]:
        if col not in wide.columns:
            wide[col] = 0
    
    wide["total_return_count"] = (
        wide[f"return_count_{positive_value}"] +
        wide[f"return_count_{baseline_value}"]
    )
    
    diff_col = f"{positive_value}_minus_{baseline_value}_share"
    ratio_col = f"{positive_value}_{baseline_value}_share_ratio"
    
    wide[diff_col] = (
        wide[f"return_share_{positive_value}"] -
        wide[f"return_share_{baseline_value}"]
    )
    
    wide[ratio_col] = (
        (wide[f"return_share_{positive_value}"] + 1e-9) /
        (wide[f"return_share_{baseline_value}"] + 1e-9)
    )
    
    wide = wide.merge(
        station_coords,
        on="end_station_name",
        how="left"
    )
    
    wide = wide[
        wide["total_return_count"] >= min_total_returns
    ].copy()
    
    wide = wide.sort_values(diff_col, ascending=False)
    
    output_path = os.path.join(OUTPUT_DIR, output_name)
    wide.to_csv(output_path, index=False)
    
    print("Saved:", output_path)
    print("Rows:", len(wide))
    
    return wide

In [ ]:
# Overall rainy vs dry
rainy_vs_dry_overall = weather_station_difference(
    station_weather_counts=station_weather_counts,
    station_coords=station_coords,
    condition_type="rain",
    positive_value="rainy",
    baseline_value="dry",
    day_type_filter=None,
    min_total_returns=50,
    output_name="weather_rainy_vs_dry_return_station_comparison_overall.csv"
)

# Weekday rainy vs dry
rainy_vs_dry_weekday = weather_station_difference(
    station_weather_counts=station_weather_counts,
    station_coords=station_coords,
    condition_type="rain",
    positive_value="rainy",
    baseline_value="dry",
    day_type_filter="weekday",
    min_total_returns=50,
    output_name="weather_rainy_vs_dry_return_station_comparison_weekday.csv"
)

# Weekend rainy vs dry
rainy_vs_dry_weekend = weather_station_difference(
    station_weather_counts=station_weather_counts,
    station_coords=station_coords,
    condition_type="rain",
    positive_value="rainy",
    baseline_value="dry",
    day_type_filter="weekend",
    min_total_returns=50,
    output_name="weather_rainy_vs_dry_return_station_comparison_weekend.csv"
)

print("Top rainy-oriented stations overall")
display(rainy_vs_dry_overall.head(20))

print("Top dry-oriented stations overall")
display(rainy_vs_dry_overall.tail(20))

In [ ]:
cold_vs_mild_overall = weather_station_difference(
    station_weather_counts=station_weather_counts,
    station_coords=station_coords,
    condition_type="cold",
    positive_value="cold",
    baseline_value="mild",
    day_type_filter=None,
    min_total_returns=50,
    output_name="weather_cold_vs_mild_return_station_comparison_overall.csv"
)

cold_vs_mild_weekday = weather_station_difference(
    station_weather_counts=station_weather_counts,
    station_coords=station_coords,
    condition_type="cold",
    positive_value="cold",
    baseline_value="mild",
    day_type_filter="weekday",
    min_total_returns=50,
    output_name="weather_cold_vs_mild_return_station_comparison_weekday.csv"
)

cold_vs_mild_weekend = weather_station_difference(
    station_weather_counts=station_weather_counts,
    station_coords=station_coords,
    condition_type="cold",
    positive_value="cold",
    baseline_value="mild",
    day_type_filter="weekend",
    min_total_returns=50,
    output_name="weather_cold_vs_mild_return_station_comparison_weekend.csv"
)

print("Top cold-oriented stations overall")
display(cold_vs_mild_overall.head(20))

In [ ]:
import folium

def make_weather_difference_map(
    diff_df,
    diff_col,
    positive_label,
    negative_label,
    output_html
):
    map_df = diff_df.dropna(subset=["end_lat", "end_lng"]).copy()
    
    # Top positive and negative stations
    top_positive = map_df.sort_values(diff_col, ascending=False).head(20)
    top_negative = map_df.sort_values(diff_col, ascending=True).head(20)
    
    plot_df = pd.concat([top_positive, top_negative], ignore_index=True)
    
    center_lat = plot_df["end_lat"].mean()
    center_lng = plot_df["end_lng"].mean()
    
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=12,
        tiles="OpenStreetMap"
    )
    
    positive_layer = folium.FeatureGroup(
        name=positive_label,
        show=True
    )
    
    negative_layer = folium.FeatureGroup(
        name=negative_label,
        show=True
    )
    
    for _, row in plot_df.iterrows():
        popup_html = f"""
        <div style="width: 330px;">
            <h4>{row['end_station_name']}</h4>
            <b>{diff_col}:</b> {row[diff_col]:.6f}<br>
            <b>Total compared returns:</b> {int(row['total_return_count'])}<br>
        </div>
        """
        
        radius = 6 + np.log1p(abs(row[diff_col]) * 1_000_000)
        
        marker = folium.CircleMarker(
            location=[row["end_lat"], row["end_lng"]],
            radius=radius,
            popup=folium.Popup(popup_html, max_width=360),
            tooltip=f"{row['end_station_name']} | {row[diff_col]:.6f}",
            fill=True,
            fill_opacity=0.75
        )
        
        if row[diff_col] > 0:
            marker.add_to(positive_layer)
        else:
            marker.add_to(negative_layer)
    
    positive_layer.add_to(m)
    negative_layer.add_to(m)
    
    folium.LayerControl(collapsed=False).add_to(m)
    
    output_path = os.path.join(OUTPUT_DIR, output_html)
    m.save(output_path)
    
    print("Saved:", output_path)
    return m

In [ ]:
rain_map = make_weather_difference_map(
    diff_df=rainy_vs_dry_overall,
    diff_col="rainy_minus_dry_share",
    positive_label="Rainy-oriented return stations",
    negative_label="Dry-oriented return stations",
    output_html="weather_rainy_vs_dry_return_station_difference_map.html"
)

rain_map

---

# Statistics / Final Figures Pipeline

Source notebook merged from: `statistics.ipynb`


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = "outputs"

station_compare = pd.read_csv(
    os.path.join(OUTPUT_DIR, "layer1_weekday_weekend_return_station_comparison.csv")
)

top_weekend = (
    station_compare
    .sort_values("weekend_minus_weekday_share", ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 6))
plt.barh(
    top_weekend["end_station_name"],
    top_weekend["weekend_minus_weekday_share"]
)
plt.xlabel("Weekend minus weekday return share")
plt.ylabel("End station")
plt.title("Top 10 Weekend-Oriented Return Stations")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "top10_weekend_oriented_stations.png"), dpi=300)
plt.show()

In [ ]:
top_weekday = (
    station_compare
    .sort_values("weekend_minus_weekday_share", ascending=True)
    .head(10)
)

plt.figure(figsize=(10, 6))
plt.barh(
    top_weekday["end_station_name"],
    top_weekday["weekend_minus_weekday_share"]
)
plt.xlabel("Weekend minus weekday return share")
plt.ylabel("End station")
plt.title("Top 10 Weekday-Oriented Return Stations")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "top10_weekday_oriented_stations.png"), dpi=300)
plt.show()

In [ ]:
service_counts = pd.read_csv(
    os.path.join(OUTPUT_DIR, "top20_end_station_osm_service_counts_250m.csv")
)

total_col = [c for c in service_counts.columns if c.startswith("total_services")][0]

top_services = service_counts.sort_values(total_col, ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(
    top_services["end_station_name"],
    top_services[total_col]
)
plt.xlabel("Number of mapped OSM services within radius")
plt.ylabel("End station")
plt.title("Top 10 Stations by Nearby OSM Service Count")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "top10_station_osm_service_count.png"), dpi=300)
plt.show()

In [ ]:
exclude_cols = {
    "end_station_name",
    "end_trip_count",
    "end_lat",
    "end_lng",
    total_col
}

service_cols = [
    c for c in service_counts.columns
    if c not in exclude_cols
    and pd.api.types.is_numeric_dtype(service_counts[c])
]

service_totals = service_counts[service_cols].sum().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
plt.bar(service_totals.index, service_totals.values)
plt.xlabel("Service category")
plt.ylabel("Total OSM services near top stations")
plt.title("Nearby OSM Service Composition Around Top End Stations")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "osm_service_category_composition.png"), dpi=300)
plt.show()

In [ ]:
rain = pd.read_csv(
    os.path.join(OUTPUT_DIR, "weather_rainy_vs_dry_return_station_comparison_overall.csv")
)

top_rainy = rain.sort_values("rainy_minus_dry_share", ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(
    top_rainy["end_station_name"],
    top_rainy["rainy_minus_dry_share"]
)
plt.xlabel("Rainy minus dry return share")
plt.ylabel("End station")
plt.title("Top 10 Rainy-Oriented Return Stations")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "top10_rainy_oriented_stations.png"), dpi=300)
plt.show()

In [ ]:
top_dry = rain.sort_values("rainy_minus_dry_share", ascending=True).head(10)

plt.figure(figsize=(10, 6))
plt.barh(
    top_dry["end_station_name"],
    top_dry["rainy_minus_dry_share"]
)
plt.xlabel("Rainy minus dry return share")
plt.ylabel("End station")
plt.title("Top 10 Dry-Oriented Return Stations")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "top10_dry_oriented_stations.png"), dpi=300)
plt.show()

In [ ]:
daily_weather = pd.read_csv("daily_trip_weather_summary.csv")

daily_weather["TAVG"] = (daily_weather["TMAX"] + daily_weather["TMIN"]) / 2

plt.figure(figsize=(8, 5))
plt.scatter(
    daily_weather["TAVG"],
    daily_weather["daily_trip_count"],
    alpha=0.7
)
plt.xlabel("Daily average temperature")
plt.ylabel("Daily trip count")
plt.title("Daily Trip Count vs Temperature")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "daily_trip_count_vs_temperature.png"), dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    daily_weather["PRCP"],
    daily_weather["daily_trip_count"],
    alpha=0.7
)
plt.xlabel("Daily precipitation")
plt.ylabel("Daily trip count")
plt.title("Daily Trip Count vs Precipitation")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "daily_trip_count_vs_precipitation.png"), dpi=300)
plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

STATION_WEATHER_COUNTS_PATH = os.path.join(
    OUTPUT_DIR,
    "station_weather_return_counts.csv"
)

STATION_COORDS_PATH = os.path.join(
    OUTPUT_DIR,
    "station_coordinates_from_cleaned_divvy.csv"
)

station_weather_counts = pd.read_csv(STATION_WEATHER_COUNTS_PATH)
station_coords = pd.read_csv(STATION_COORDS_PATH)

station_weather_counts["end_station_name"] = (
    station_weather_counts["end_station_name"]
    .astype(str)
    .str.strip()
)

station_coords["end_station_name"] = (
    station_coords["end_station_name"]
    .astype(str)
    .str.strip()
)

# Merge coordinates
station_weather_geo = station_weather_counts.merge(
    station_coords,
    on="end_station_name",
    how="left"
)

station_weather_geo = station_weather_geo.dropna(
    subset=["end_lat", "end_lng"]
).copy()

# Calculate return share inside each day_type + weather condition
station_weather_geo["total_returns_in_group"] = (
    station_weather_geo
    .groupby(["condition_type", "day_type", "condition_value"])["return_count"]
    .transform("sum")
)

station_weather_geo["return_share"] = (
    station_weather_geo["return_count"] /
    station_weather_geo["total_returns_in_group"]
)

output_csv = os.path.join(
    OUTPUT_DIR,
    "station_weather_geo_with_return_share.csv"
)

station_weather_geo.to_csv(output_csv, index=False)

print("Saved:", output_csv)
display(station_weather_geo.head())
print(station_weather_geo[["condition_type", "condition_value", "day_type"]].drop_duplicates())

In [ ]:
top_station_rows = []

for (condition_type, condition_value, day_type), group in station_weather_geo.groupby(
    ["condition_type", "condition_value", "day_type"]
):
    top_group = (
        group
        .sort_values("return_share", ascending=False)
        .head(20)
        .copy()
    )
    
    top_group["rank_in_group"] = range(1, len(top_group) + 1)
    
    top_station_rows.append(top_group)

top_weather_stations = pd.concat(top_station_rows, ignore_index=True)

top_weather_stations = top_weather_stations[[
    "condition_type",
    "condition_value",
    "day_type",
    "rank_in_group",
    "end_station_name",
    "return_count",
    "return_share",
    "total_returns_in_group",
    "end_lat",
    "end_lng"
]]

output_csv = os.path.join(
    OUTPUT_DIR,
    "top20_end_stations_by_daytype_weather.csv"
)

top_weather_stations.to_csv(output_csv, index=False)

print("Saved:", output_csv)
display(top_weather_stations.head(30))

In [ ]:
FIG_DIR = os.path.join(OUTPUT_DIR, "weather_station_figures")
os.makedirs(FIG_DIR, exist_ok=True)

for (condition_type, condition_value, day_type), group in station_weather_geo.groupby(
    ["condition_type", "condition_value", "day_type"]
):
    top10 = (
        group
        .sort_values("return_share", ascending=False)
        .head(10)
        .copy()
    )
    
    if len(top10) == 0:
        continue
    
    plt.figure(figsize=(10, 6))
    plt.barh(
        top10["end_station_name"],
        top10["return_share"]
    )
    
    plt.xlabel("Return share within this day/weather group")
    plt.ylabel("End station")
    plt.title(
        f"Top 10 Return Stations: {day_type.capitalize()} / {condition_value.capitalize()} "
        f"({condition_type})"
    )
    plt.gca().invert_yaxis()
    plt.tight_layout()
    
    safe_name = f"top10_{day_type}_{condition_type}_{condition_value}_return_stations.png"
    output_fig = os.path.join(FIG_DIR, safe_name)
    
    plt.savefig(output_fig, dpi=300)
    plt.show()
    
    print("Saved:", output_fig)

In [ ]:
# Base map center
center_lat = station_weather_geo["end_lat"].mean()
center_lng = station_weather_geo["end_lng"].mean()

m_weather_heat = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=12,
    tiles="OpenStreetMap"
)

# Add one heatmap layer for each day_type + weather condition
for (condition_type, condition_value, day_type), group in station_weather_geo.groupby(
    ["condition_type", "condition_value", "day_type"]
):
    layer_name = f"Heatmap: {day_type} | {condition_value} ({condition_type})"
    
    heat_layer = folium.FeatureGroup(
        name=layer_name,
        show=False
    )
    
    # Use return_count as heatmap weight
    heat_data = group[
        ["end_lat", "end_lng", "return_count"]
    ].dropna().values.tolist()
    
    if len(heat_data) == 0:
        continue
    
    HeatMap(
        heat_data,
        radius=18,
        blur=15,
        max_zoom=13
    ).add_to(heat_layer)
    
    heat_layer.add_to(m_weather_heat)

folium.LayerControl(collapsed=False).add_to(m_weather_heat)

output_map = os.path.join(
    OUTPUT_DIR,
    "weather_daytype_end_station_heatmaps.html"
)

m_weather_heat.save(output_map)

print("Saved:", output_map)
m_weather_heat

In [ ]:
weather_daytype_diff_rows = []

for (condition_type, condition_value), group in station_weather_geo.groupby(
    ["condition_type", "condition_value"]
):
    temp = group.pivot_table(
        index="end_station_name",
        columns="day_type",
        values=["return_count", "return_share"],
        fill_value=0
    ).reset_index()
    
    temp.columns = [
        "_".join([str(x) for x in col if x != ""])
        for col in temp.columns
    ]
    
    for col in [
        "return_count_weekday",
        "return_count_weekend",
        "return_share_weekday",
        "return_share_weekend"
    ]:
        if col not in temp.columns:
            temp[col] = 0
    
    temp["condition_type"] = condition_type
    temp["condition_value"] = condition_value
    
    temp["weekend_minus_weekday_share_under_weather"] = (
        temp["return_share_weekend"] -
        temp["return_share_weekday"]
    )
    
    temp["total_return_count_under_weather"] = (
        temp["return_count_weekday"] +
        temp["return_count_weekend"]
    )
    
    temp = temp.merge(
        station_coords,
        on="end_station_name",
        how="left"
    )
    
    weather_daytype_diff_rows.append(temp)

weather_daytype_diff = pd.concat(weather_daytype_diff_rows, ignore_index=True)

# Filter small stations
weather_daytype_diff = weather_daytype_diff[
    weather_daytype_diff["total_return_count_under_weather"] >= 50
].copy()

output_csv = os.path.join(
    OUTPUT_DIR,
    "weather_specific_weekend_minus_weekday_station_difference.csv"
)

weather_daytype_diff.to_csv(output_csv, index=False)

print("Saved:", output_csv)
display(weather_daytype_diff.head())

In [ ]:
DIFF_FIG_DIR = os.path.join(OUTPUT_DIR, "weather_weekday_weekend_difference_figures")
os.makedirs(DIFF_FIG_DIR, exist_ok=True)

for (condition_type, condition_value), group in weather_daytype_diff.groupby(
    ["condition_type", "condition_value"]
):
    # Top weekend-oriented under this weather
    top_weekend = (
        group
        .sort_values("weekend_minus_weekday_share_under_weather", ascending=False)
        .head(10)
        .copy()
    )
    
    plt.figure(figsize=(10, 6))
    plt.barh(
        top_weekend["end_station_name"],
        top_weekend["weekend_minus_weekday_share_under_weather"]
    )
    plt.xlabel(f"Weekend minus weekday return share under {condition_value}")
    plt.ylabel("End station")
    plt.title(f"Weekend-Oriented Return Stations Under {condition_value.capitalize()} Weather")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    
    output_fig = os.path.join(
        DIFF_FIG_DIR,
        f"top10_weekend_oriented_under_{condition_type}_{condition_value}.png"
    )
    
    plt.savefig(output_fig, dpi=300)
    plt.show()
    print("Saved:", output_fig)
    
    
    # Top weekday-oriented under this weather
    top_weekday = (
        group
        .sort_values("weekend_minus_weekday_share_under_weather", ascending=True)
        .head(10)
        .copy()
    )
    
    plt.figure(figsize=(10, 6))
    plt.barh(
        top_weekday["end_station_name"],
        top_weekday["weekend_minus_weekday_share_under_weather"]
    )
    plt.xlabel(f"Weekend minus weekday return share under {condition_value}")
    plt.ylabel("End station")
    plt.title(f"Weekday-Oriented Return Stations Under {condition_value.capitalize()} Weather")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    
    output_fig = os.path.join(
        DIFF_FIG_DIR,
        f"top10_weekday_oriented_under_{condition_type}_{condition_value}.png"
    )
    
    plt.savefig(output_fig, dpi=300)
    plt.show()
    print("Saved:", output_fig)

In [ ]:
m_weather_diff = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=12,
    tiles="OpenStreetMap"
)

for (condition_type, condition_value), group in weather_daytype_diff.groupby(
    ["condition_type", "condition_value"]
):
    top_weekend = (
        group
        .sort_values("weekend_minus_weekday_share_under_weather", ascending=False)
        .head(20)
        .copy()
    )
    
    top_weekday = (
        group
        .sort_values("weekend_minus_weekday_share_under_weather", ascending=True)
        .head(20)
        .copy()
    )
    
    weekend_layer = folium.FeatureGroup(
        name=f"Weekend-oriented under {condition_value} ({condition_type})",
        show=False
    )
    
    weekday_layer = folium.FeatureGroup(
        name=f"Weekday-oriented under {condition_value} ({condition_type})",
        show=False
    )
    
    for _, row in top_weekend.iterrows():
        popup_html = f"""
        <div style="width: 340px;">
            <h4>{row['end_station_name']}</h4>
            <b>Weather condition:</b> {condition_value} ({condition_type})<br>
            <b>Weekday returns:</b> {int(row['return_count_weekday'])}<br>
            <b>Weekend returns:</b> {int(row['return_count_weekend'])}<br>
            <b>Weekday share:</b> {row['return_share_weekday']:.6f}<br>
            <b>Weekend share:</b> {row['return_share_weekend']:.6f}<br>
            <b>Weekend - weekday share:</b> {row['weekend_minus_weekday_share_under_weather']:.6f}<br>
            <br>
            <i>Positive value means this station is relatively more weekend-oriented under this weather condition.</i>
        </div>
        """
        
        radius = 6 + np.log1p(abs(row["weekend_minus_weekday_share_under_weather"]) * 1_000_000)
        
        folium.CircleMarker(
            location=[row["end_lat"], row["end_lng"]],
            radius=radius,
            popup=folium.Popup(popup_html, max_width=370),
            tooltip=f"Weekend-oriented under {condition_value}: {row['end_station_name']}",
            fill=True,
            fill_opacity=0.75
        ).add_to(weekend_layer)
    
    for _, row in top_weekday.iterrows():
        popup_html = f"""
        <div style="width: 340px;">
            <h4>{row['end_station_name']}</h4>
            <b>Weather condition:</b> {condition_value} ({condition_type})<br>
            <b>Weekday returns:</b> {int(row['return_count_weekday'])}<br>
            <b>Weekend returns:</b> {int(row['return_count_weekend'])}<br>
            <b>Weekday share:</b> {row['return_share_weekday']:.6f}<br>
            <b>Weekend share:</b> {row['return_share_weekend']:.6f}<br>
            <b>Weekend - weekday share:</b> {row['weekend_minus_weekday_share_under_weather']:.6f}<br>
            <br>
            <i>Negative value means this station is relatively more weekday-oriented under this weather condition.</i>
        </div>
        """
        
        radius = 6 + np.log1p(abs(row["weekend_minus_weekday_share_under_weather"]) * 1_000_000)
        
        folium.CircleMarker(
            location=[row["end_lat"], row["end_lng"]],
            radius=radius,
            popup=folium.Popup(popup_html, max_width=370),
            tooltip=f"Weekday-oriented under {condition_value}: {row['end_station_name']}",
            fill=True,
            fill_opacity=0.75
        ).add_to(weekday_layer)
    
    weekend_layer.add_to(m_weather_diff)
    weekday_layer.add_to(m_weather_diff)

folium.LayerControl(collapsed=False).add_to(m_weather_diff)

output_map = os.path.join(
    OUTPUT_DIR,
    "weather_specific_weekday_weekend_difference_station_map.html"
)

m_weather_diff.save(output_map)

print("Saved:", output_map)
m_weather_diff

---

# Validation + Weather Validation Pipeline

Source notebook merged from: `validation_with_weather.ipynb`


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

STATION_COMPARE_PATH = os.path.join(
    OUTPUT_DIR,
    "layer1_weekday_weekend_return_station_comparison.csv"
)

SERVICE_COUNTS_250_PATH = os.path.join(
    OUTPUT_DIR,
    "top20_end_station_osm_service_counts_250m.csv"
)

station_compare = pd.read_csv(STATION_COMPARE_PATH)
service_counts_250 = pd.read_csv(SERVICE_COUNTS_250_PATH)

station_compare["end_station_name"] = station_compare["end_station_name"].astype(str).str.strip()
service_counts_250["end_station_name"] = service_counts_250["end_station_name"].astype(str).str.strip()


def identify_service_columns(service_df):
    """
    Identify numeric OSM service category columns.
    """
    non_service_cols = {
        "end_station_name",
        "end_trip_count",
        "station_end_trip_count",
        "end_lat",
        "end_lng",
        "station_lat",
        "station_lng",
        "search_radius_m",
        "matched_address"
    }
    
    service_cols = []
    
    for col in service_df.columns:
        if col in non_service_cols:
            continue
        if col.startswith("total_services"):
            continue
        if pd.api.types.is_numeric_dtype(service_df[col]):
            service_cols.append(col)
    
    return service_cols


def prepare_profile_data(station_compare, service_counts):
    """
    Merge station return comparison with OSM service counts.
    """
    df = station_compare.merge(
        service_counts,
        on="end_station_name",
        how="inner",
        suffixes=("", "_service")
    )
    
    service_cols = identify_service_columns(service_counts)
    
    return df, service_cols


def weighted_service_profile(df, service_cols, weight_col, normalize_station_profile=True):
    """
    Compute weighted service profile.
    
    If normalize_station_profile=True:
    each station's service vector is converted into proportions first.
    
    Then station profiles are averaged using return counts as weights.
    """
    service_matrix = df[service_cols].fillna(0).astype(float).copy()
    
    if normalize_station_profile:
        row_sum = service_matrix.sum(axis=1)
        service_matrix = service_matrix.div(row_sum.replace(0, np.nan), axis=0).fillna(0)
    
    weights = df[weight_col].fillna(0).astype(float).values
    
    if weights.sum() == 0:
        return pd.Series(np.zeros(len(service_cols)), index=service_cols)
    
    weights = weights / weights.sum()
    
    profile = (service_matrix.values * weights[:, None]).sum(axis=0)
    
    return pd.Series(profile, index=service_cols)


profile_df_250, service_cols_250 = prepare_profile_data(
    station_compare,
    service_counts_250
)

print("Merged profile dataset:", profile_df_250.shape)
print("Service columns:", service_cols_250)

display(profile_df_250.head())

In [ ]:
# =========================
# Baseline: station count ranking only
# =========================

baseline_weekday_top20 = (
    station_compare
    .sort_values("return_count_weekday", ascending=False)
    .head(20)
    .copy()
)

baseline_weekend_top20 = (
    station_compare
    .sort_values("return_count_weekend", ascending=False)
    .head(20)
    .copy()
)

baseline_weekday_top20.to_csv(
    os.path.join(OUTPUT_DIR, "baseline_top20_weekday_return_stations.csv"),
    index=False
)

baseline_weekend_top20.to_csv(
    os.path.join(OUTPUT_DIR, "baseline_top20_weekend_return_stations.csv"),
    index=False
)

print("Baseline top 20 weekday return stations")
display(baseline_weekday_top20[[
    "end_station_name",
    "return_count_weekday",
    "return_share_weekday",
    "end_lat",
    "end_lng"
]])

print("Baseline top 20 weekend return stations")
display(baseline_weekend_top20[[
    "end_station_name",
    "return_count_weekend",
    "return_share_weekend",
    "end_lat",
    "end_lng"
]])

In [ ]:
# =========================
# Profiling model: weighted service profile
# =========================

weekday_profile_250 = weighted_service_profile(
    profile_df_250,
    service_cols_250,
    weight_col="return_count_weekday",
    normalize_station_profile=True
)

weekend_profile_250 = weighted_service_profile(
    profile_df_250,
    service_cols_250,
    weight_col="return_count_weekend",
    normalize_station_profile=True
)

service_profile_comparison_250 = pd.DataFrame({
    "service_category": service_cols_250,
    "weekday_profile": weekday_profile_250.values,
    "weekend_profile": weekend_profile_250.values
})

service_profile_comparison_250["weekend_minus_weekday"] = (
    service_profile_comparison_250["weekend_profile"] -
    service_profile_comparison_250["weekday_profile"]
)

service_profile_comparison_250 = service_profile_comparison_250.sort_values(
    "weekend_minus_weekday",
    ascending=False
)

output_csv = os.path.join(
    OUTPUT_DIR,
    "validation1_weekday_weekend_service_profile_comparison_250m.csv"
)

service_profile_comparison_250.to_csv(output_csv, index=False)

print("Saved:", output_csv)
display(service_profile_comparison_250)

In [ ]:
plot_df = service_profile_comparison_250.sort_values("weekend_minus_weekday")

plt.figure(figsize=(9, 6))
plt.barh(
    plot_df["service_category"],
    plot_df["weekend_minus_weekday"]
)

plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Weekend profile minus weekday profile")
plt.ylabel("Service category")
plt.title("Validation 1: Weekday vs Weekend Service Profile Difference")
plt.tight_layout()

output_fig = os.path.join(
    OUTPUT_DIR,
    "validation1_weekday_weekend_service_profile_difference.png"
)

plt.savefig(output_fig, dpi=300)
plt.show()

print("Saved:", output_fig)

In [ ]:
# =========================
# Validation 2: Random shuffle test
# =========================

df = profile_df_250.copy()
service_cols = service_cols_250.copy()

# Station service profiles
service_matrix = df[service_cols].fillna(0).astype(float)
row_sum = service_matrix.sum(axis=1)
station_service_profile = service_matrix.div(row_sum.replace(0, np.nan), axis=0).fillna(0)

def compute_profile_from_weights(weights):
    weights = np.asarray(weights, dtype=float)
    
    if weights.sum() == 0:
        return np.zeros(len(service_cols))
    
    weights = weights / weights.sum()
    return (station_service_profile.values * weights[:, None]).sum(axis=0)

# Observed difference
weekday_weights_obs = df["return_count_weekday"].fillna(0).astype(int).values
weekend_weights_obs = df["return_count_weekend"].fillna(0).astype(int).values

weekday_profile_obs = compute_profile_from_weights(weekday_weights_obs)
weekend_profile_obs = compute_profile_from_weights(weekend_weights_obs)

# L1 distance between profiles
observed_diff = np.abs(weekend_profile_obs - weekday_profile_obs).sum()

print("Observed weekday-weekend service profile difference:", observed_diff)

# Permutation test
rng = np.random.default_rng(42)
n_perm = 1000

station_total_returns = weekday_weights_obs + weekend_weights_obs

total_weekend_returns = weekend_weights_obs.sum()
total_all_returns = station_total_returns.sum()

p_weekend = total_weekend_returns / total_all_returns

perm_diffs = []

for i in range(n_perm):
    # Approximate random shuffle:
    # each station's total returns are randomly split into weekday/weekend
    shuffled_weekend = rng.binomial(station_total_returns, p_weekend)
    shuffled_weekday = station_total_returns - shuffled_weekend
    
    shuffled_weekday_profile = compute_profile_from_weights(shuffled_weekday)
    shuffled_weekend_profile = compute_profile_from_weights(shuffled_weekend)
    
    shuffled_diff = np.abs(
        shuffled_weekend_profile - shuffled_weekday_profile
    ).sum()
    
    perm_diffs.append(shuffled_diff)

perm_diffs = np.array(perm_diffs)

p_value = (np.sum(perm_diffs >= observed_diff) + 1) / (n_perm + 1)

shuffle_result = pd.DataFrame({
    "observed_profile_difference": [observed_diff],
    "mean_shuffled_difference": [perm_diffs.mean()],
    "std_shuffled_difference": [perm_diffs.std()],
    "p_value": [p_value],
    "n_permutations": [n_perm],
    "interpretation": [
        "Small p-value means observed weekday/weekend service-profile difference is larger than most random label shuffles."
    ]
})

output_csv = os.path.join(
    OUTPUT_DIR,
    "validation2_weekday_weekend_shuffle_test.csv"
)

shuffle_result.to_csv(output_csv, index=False)

print("Saved:", output_csv)
display(shuffle_result)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(perm_diffs, bins=40, alpha=0.8)
plt.axvline(observed_diff, linestyle="--", linewidth=2)

plt.xlabel("Shuffled weekday/weekend service-profile difference")
plt.ylabel("Frequency")
plt.title("Validation 2: Random Shuffle Test")
plt.tight_layout()

output_fig = os.path.join(
    OUTPUT_DIR,
    "validation2_shuffle_test_weekday_weekend_service_profile.png"
)

plt.savefig(output_fig, dpi=300)
plt.show()

print("Saved:", output_fig)

In [ ]:
# If needed:
# %pip install osmnx geopandas shapely

import time
import osmnx as ox
import geopandas as gpd
from shapely.geometry import Point

MAX_RADIUS_M = 500

OSM_DETAIL_500_PATH = os.path.join(
    OUTPUT_DIR,
    "validation3_top20_osm_services_max500m_detail.csv"
)

# Use top 20 high-volume return stations from the station comparison table
top20_for_radius = (
    station_compare
    .sort_values("total_return_count", ascending=False)
    .head(20)
    .copy()
)

top20_for_radius = top20_for_radius.dropna(subset=["end_lat", "end_lng"]).copy()

top20_for_radius = top20_for_radius[[
    "end_station_name",
    "total_return_count",
    "end_lat",
    "end_lng"
]].rename(columns={
    "total_return_count": "end_trip_count"
})

top20_for_radius["end_station_name"] = top20_for_radius["end_station_name"].astype(str).str.strip()

display(top20_for_radius)


OSM_TAGS = {
    "amenity": True,
    "shop": True,
    "tourism": True,
    "office": True,
    "leisure": True,
    "public_transport": True,
    "railway": ["station", "subway_entrance", "tram_stop"],
    "healthcare": True
}


def get_tag(row, col):
    if col not in row.index:
        return ""
    value = row[col]
    if pd.isna(value):
        return ""
    return str(value).lower()


def classify_osm_service(row):
    amenity = get_tag(row, "amenity")
    shop = get_tag(row, "shop")
    tourism = get_tag(row, "tourism")
    office = get_tag(row, "office")
    leisure = get_tag(row, "leisure")
    public_transport = get_tag(row, "public_transport")
    railway = get_tag(row, "railway")
    healthcare = get_tag(row, "healthcare")

    food_drink = {
        "restaurant", "cafe", "bar", "pub", "fast_food",
        "food_court", "ice_cream", "biergarten"
    }

    transit = {
        "bus_station", "ferry_terminal", "taxi"
    }

    education = {
        "school", "college", "university", "library",
        "kindergarten"
    }

    health = {
        "hospital", "clinic", "doctors", "dentist", "pharmacy"
    }

    if amenity in food_drink:
        return "food_drink"

    if amenity in transit or public_transport != "" or railway in ["station", "subway_entrance", "tram_stop"]:
        return "transit"

    if shop != "":
        return "retail"

    if tourism != "":
        return "tourism"

    if office != "":
        return "office"

    if amenity in health or healthcare != "":
        return "health"

    if leisure != "":
        return "recreation"

    if amenity in education:
        return "education"

    if amenity != "":
        return "other_amenity"

    return "other_service"


if os.path.exists(OSM_DETAIL_500_PATH):
    print("Cached OSM detail file already exists. Reading:")
    print(OSM_DETAIL_500_PATH)
    osm_detail_500 = pd.read_csv(OSM_DETAIL_500_PATH)

else:
    all_service_rows = []

    for _, station in top20_for_radius.iterrows():
        station_name = station["end_station_name"]
        lat = station["end_lat"]
        lng = station["end_lng"]

        print(f"Querying OSM services within {MAX_RADIUS_M}m around: {station_name}")

        try:
            gdf = ox.features_from_point(
                center_point=(lat, lng),
                tags=OSM_TAGS,
                dist=MAX_RADIUS_M
            )

            if gdf.empty:
                print("  No services found.")
                continue

            gdf = gdf.reset_index()
            gdf = gdf[gdf.geometry.notna()].copy()
            gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs="EPSG:4326")

            # Classify service category
            gdf["service_category"] = gdf.apply(classify_osm_service, axis=1)

            # Project to meters for distance calculation
            gdf_proj = gdf.to_crs("EPSG:26916")
            service_centroids_proj = gdf_proj.geometry.centroid

            station_point_wgs = gpd.GeoSeries(
                [Point(lng, lat)],
                crs="EPSG:4326"
            )

            station_point_proj = station_point_wgs.to_crs("EPSG:26916").iloc[0]

            distances_m = service_centroids_proj.distance(station_point_proj)

            # Convert centroids back to WGS84
            service_centroids_wgs = gpd.GeoSeries(
                service_centroids_proj,
                crs="EPSG:26916"
            ).to_crs("EPSG:4326")

            gdf["service_lng"] = service_centroids_wgs.x.values
            gdf["service_lat"] = service_centroids_wgs.y.values
            gdf["distance_m"] = distances_m.values

            gdf["end_station_name"] = station_name
            gdf["station_end_trip_count"] = station["end_trip_count"]
            gdf["station_lat"] = lat
            gdf["station_lng"] = lng

            useful_cols = [
                "end_station_name",
                "station_end_trip_count",
                "station_lat",
                "station_lng",
                "service_category",
                "service_lat",
                "service_lng",
                "distance_m",
                "name",
                "amenity",
                "shop",
                "tourism",
                "office",
                "leisure",
                "public_transport",
                "railway",
                "healthcare"
            ]

            existing_cols = [c for c in useful_cols if c in gdf.columns]

            all_service_rows.append(gdf[existing_cols].copy())

            print(f"  Found {len(gdf)} OSM features.")
            time.sleep(1)

        except Exception as e:
            print(f"  Error for {station_name}: {e}")

    if len(all_service_rows) > 0:
        osm_detail_500 = pd.concat(all_service_rows, ignore_index=True)
    else:
        osm_detail_500 = pd.DataFrame()

    osm_detail_500.to_csv(OSM_DETAIL_500_PATH, index=False)

    print("Saved:")
    print(OSM_DETAIL_500_PATH)

print("OSM detail rows:", len(osm_detail_500))
display(osm_detail_500.head())

In [ ]:
RADIUS_LIST = [100, 250, 500]

for radius in RADIUS_LIST:
    temp = osm_detail_500[
        osm_detail_500["distance_m"] <= radius
    ].copy()
    
    if temp.empty:
        print(f"No services within {radius}m.")
        continue
    
    service_summary = (
        temp
        .groupby(["end_station_name", "service_category"])
        .size()
        .reset_index(name="service_count")
    )
    
    service_summary_wide = (
        service_summary
        .pivot_table(
            index="end_station_name",
            columns="service_category",
            values="service_count",
            fill_value=0
        )
        .reset_index()
    )
    
    station_info = top20_for_radius[[
        "end_station_name",
        "end_trip_count",
        "end_lat",
        "end_lng"
    ]].copy()
    
    service_counts_radius = station_info.merge(
        service_summary_wide,
        on="end_station_name",
        how="left"
    ).fillna(0)
    
    service_cols_radius = [
        c for c in service_counts_radius.columns
        if c not in ["end_station_name", "end_trip_count", "end_lat", "end_lng"]
    ]
    
    service_counts_radius[f"total_services_{radius}m"] = (
        service_counts_radius[service_cols_radius].sum(axis=1)
    )
    
    output_csv = os.path.join(
        OUTPUT_DIR,
        f"top20_end_station_osm_service_counts_{radius}m.csv"
    )
    
    service_counts_radius.to_csv(output_csv, index=False)
    
    print("Saved:", output_csv)
    display(service_counts_radius.head())

In [ ]:
radius_files = {
    100: os.path.join(OUTPUT_DIR, "top20_end_station_osm_service_counts_100m.csv"),
    250: os.path.join(OUTPUT_DIR, "top20_end_station_osm_service_counts_250m.csv"),
    500: os.path.join(OUTPUT_DIR, "top20_end_station_osm_service_counts_500m.csv")
}

sensitivity_rows = []
profile_detail_rows = []

for radius, path in radius_files.items():
    if not os.path.exists(path):
        print("Missing:", path)
        continue
    
    service_counts_r = pd.read_csv(path)
    service_counts_r["end_station_name"] = service_counts_r["end_station_name"].astype(str).str.strip()
    
    df_r, service_cols_r = prepare_profile_data(
        station_compare,
        service_counts_r
    )
    
    weekday_profile_r = weighted_service_profile(
        df_r,
        service_cols_r,
        weight_col="return_count_weekday",
        normalize_station_profile=True
    )
    
    weekend_profile_r = weighted_service_profile(
        df_r,
        service_cols_r,
        weight_col="return_count_weekend",
        normalize_station_profile=True
    )
    
    diff_r = weekend_profile_r - weekday_profile_r
    
    profile_difference_l1 = np.abs(diff_r).sum()
    
    most_weekend_service = diff_r.sort_values(ascending=False).index[0]
    most_weekday_service = diff_r.sort_values(ascending=True).index[0]
    
    sensitivity_rows.append({
        "radius_m": radius,
        "profile_difference_l1": profile_difference_l1,
        "most_weekend_associated_service": most_weekend_service,
        "most_weekend_associated_value": diff_r[most_weekend_service],
        "most_weekday_associated_service": most_weekday_service,
        "most_weekday_associated_value": diff_r[most_weekday_service],
        "n_stations_used": len(df_r)
    })
    
    for service in diff_r.index:
        profile_detail_rows.append({
            "radius_m": radius,
            "service_category": service,
            "weekday_profile": weekday_profile_r[service],
            "weekend_profile": weekend_profile_r[service],
            "weekend_minus_weekday": diff_r[service]
        })

sensitivity_table = pd.DataFrame(sensitivity_rows)
profile_detail_table = pd.DataFrame(profile_detail_rows)

sensitivity_csv = os.path.join(
    OUTPUT_DIR,
    "validation3_radius_sensitivity_summary.csv"
)

profile_detail_csv = os.path.join(
    OUTPUT_DIR,
    "validation3_radius_sensitivity_service_details.csv"
)

sensitivity_table.to_csv(sensitivity_csv, index=False)
profile_detail_table.to_csv(profile_detail_csv, index=False)

print("Saved:", sensitivity_csv)
print("Saved:", profile_detail_csv)

display(sensitivity_table)
display(profile_detail_table.head())

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(
    sensitivity_table["radius_m"],
    sensitivity_table["profile_difference_l1"],
    marker="o"
)

plt.xlabel("OSM search radius (m)")
plt.ylabel("Weekday/weekend service-profile difference")
plt.title("Validation 3: Radius Sensitivity Test")
plt.tight_layout()

output_fig = os.path.join(
    OUTPUT_DIR,
    "validation3_radius_sensitivity_profile_difference.png"
)

plt.savefig(output_fig, dpi=300)
plt.show()

print("Saved:", output_fig)

In [ ]:
pivot_detail = profile_detail_table.pivot_table(
    index="service_category",
    columns="radius_m",
    values="weekend_minus_weekday",
    fill_value=0
)

display(pivot_detail)

pivot_detail.to_csv(
    os.path.join(OUTPUT_DIR, "validation3_radius_sensitivity_service_pivot.csv")
)

plt.figure(figsize=(10, 6))

for service in pivot_detail.index:
    plt.plot(
        pivot_detail.columns,
        pivot_detail.loc[service],
        marker="o",
        label=service
    )

plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("OSM search radius (m)")
plt.ylabel("Weekend minus weekday service profile")
plt.title("Service Profile Difference Across Radius Choices")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

output_fig = os.path.join(
    OUTPUT_DIR,
    "validation3_service_difference_by_radius.png"
)

plt.savefig(output_fig, dpi=300)
plt.show()

print("Saved:", output_fig)

In [ ]:
# =========================
# Weather Validation Setup
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

STATION_WEATHER_COUNTS_PATH = os.path.join(
    OUTPUT_DIR,
    "station_weather_return_counts.csv"
)

SERVICE_COUNTS_PATH = os.path.join(
    OUTPUT_DIR,
    "top20_end_station_osm_service_counts_250m.csv"
)

if not os.path.exists(STATION_WEATHER_COUNTS_PATH):
    raise FileNotFoundError(
        f"Missing {STATION_WEATHER_COUNTS_PATH}. "
        "Run the weather section in openmap.ipynb first."
    )

if not os.path.exists(SERVICE_COUNTS_PATH):
    raise FileNotFoundError(
        f"Missing {SERVICE_COUNTS_PATH}. "
        "Run the OSM service section first."
    )

station_weather_counts = pd.read_csv(STATION_WEATHER_COUNTS_PATH)
service_counts = pd.read_csv(SERVICE_COUNTS_PATH)

station_weather_counts["end_station_name"] = (
    station_weather_counts["end_station_name"]
    .astype(str)
    .str.strip()
)

service_counts["end_station_name"] = (
    service_counts["end_station_name"]
    .astype(str)
    .str.strip()
)

print("station_weather_counts:", station_weather_counts.shape)
print("service_counts:", service_counts.shape)

print("Available weather groups:")
display(
    station_weather_counts[
        ["condition_type", "condition_value", "day_type"]
    ].drop_duplicates().sort_values(
        ["condition_type", "condition_value", "day_type"]
    )
)

display(station_weather_counts.head())
display(service_counts.head())

In [ ]:
# =========================
# Identify service columns and define profile function
# =========================

def identify_service_columns_for_weather(service_df):
    non_service_cols = {
        "end_station_name",
        "end_trip_count",
        "station_end_trip_count",
        "end_lat",
        "end_lng",
        "station_lat",
        "station_lng",
        "matched_address",
        "search_radius_m"
    }

    service_cols = []

    for col in service_df.columns:
        if col in non_service_cols:
            continue
        if str(col).startswith("total_services"):
            continue
        if pd.api.types.is_numeric_dtype(service_df[col]):
            service_cols.append(col)

    return service_cols


service_cols_weather = identify_service_columns_for_weather(service_counts)

print("Service columns used:")
print(service_cols_weather)


def compute_weather_group_profile(
    station_weather_counts,
    service_counts,
    service_cols,
    condition_type,
    condition_value,
    day_type
):
    """
    Compute weighted destination-service profile for one weather/day group.

    Weight = station return count within that weather/day group.
    Station vector = normalized OSM service profile.
    """

    temp = station_weather_counts[
        (station_weather_counts["condition_type"] == condition_type) &
        (station_weather_counts["condition_value"] == condition_value) &
        (station_weather_counts["day_type"] == day_type)
    ].copy()

    if temp.empty:
        return pd.Series(np.zeros(len(service_cols)), index=service_cols)

    temp = (
        temp
        .groupby("end_station_name", as_index=False)
        .agg(return_count=("return_count", "sum"))
    )

    temp = temp.merge(
        service_counts[["end_station_name"] + service_cols],
        on="end_station_name",
        how="inner"
    )

    if temp.empty:
        return pd.Series(np.zeros(len(service_cols)), index=service_cols)

    service_matrix = temp[service_cols].fillna(0).astype(float)

    # Normalize each station's service vector into a service profile
    row_sum = service_matrix.sum(axis=1)
    station_service_profile = (
        service_matrix
        .div(row_sum.replace(0, np.nan), axis=0)
        .fillna(0)
    )

    weights = temp["return_count"].fillna(0).astype(float).values

    if weights.sum() == 0:
        return pd.Series(np.zeros(len(service_cols)), index=service_cols)

    weights = weights / weights.sum()

    profile = (station_service_profile.values * weights[:, None]).sum(axis=0)

    return pd.Series(profile, index=service_cols)

In [ ]:
# =========================
# Compute service profiles for weather/day groups
# =========================

weather_groups = [
    ("rain", "dry", "weekday", "dry_weekday"),
    ("rain", "rainy", "weekday", "rainy_weekday"),
    ("rain", "dry", "weekend", "dry_weekend"),
    ("rain", "rainy", "weekend", "rainy_weekend"),

    ("cold", "mild", "weekday", "mild_weekday"),
    ("cold", "cold", "weekday", "cold_weekday"),
    ("cold", "mild", "weekend", "mild_weekend"),
    ("cold", "cold", "weekend", "cold_weekend")
]

profile_rows = []

for condition_type, condition_value, day_type, group_name in weather_groups:
    profile = compute_weather_group_profile(
        station_weather_counts=station_weather_counts,
        service_counts=service_counts,
        service_cols=service_cols_weather,
        condition_type=condition_type,
        condition_value=condition_value,
        day_type=day_type
    )

    for service in service_cols_weather:
        profile_rows.append({
            "group_name": group_name,
            "condition_type": condition_type,
            "condition_value": condition_value,
            "day_type": day_type,
            "service_category": service,
            "profile_value": profile[service]
        })

weather_group_profiles = pd.DataFrame(profile_rows)

output_csv = os.path.join(
    OUTPUT_DIR,
    "validation_weather_group_service_profiles.csv"
)

weather_group_profiles.to_csv(output_csv, index=False)

print("Saved:", output_csv)
display(weather_group_profiles.head(20))

In [ ]:
# =========================
# Plot weather group service profile heatmap
# =========================

profile_pivot = weather_group_profiles.pivot_table(
    index="service_category",
    columns="group_name",
    values="profile_value",
    fill_value=0
)

ordered_cols = [
    "dry_weekday",
    "rainy_weekday",
    "dry_weekend",
    "rainy_weekend",
    "mild_weekday",
    "cold_weekday",
    "mild_weekend",
    "cold_weekend"
]

ordered_cols = [c for c in ordered_cols if c in profile_pivot.columns]
profile_pivot = profile_pivot[ordered_cols]

plt.figure(figsize=(11, 7))
plt.imshow(profile_pivot.values, aspect="auto")

plt.xticks(
    ticks=np.arange(len(profile_pivot.columns)),
    labels=profile_pivot.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    ticks=np.arange(len(profile_pivot.index)),
    labels=profile_pivot.index
)

plt.colorbar(label="Weighted service profile value")
plt.title("Weather Validation: Destination-Service Profiles by Weather and Day Type")
plt.tight_layout()

output_fig = os.path.join(
    OUTPUT_DIR,
    "validation_weather_group_service_profile_heatmap.png"
)

plt.savefig(output_fig, dpi=300)
plt.show()

print("Saved:", output_fig)
display(profile_pivot)

In [ ]:
# =========================
# Rainy vs dry profile differences
# =========================

def get_weather_profile(group_name):
    temp = weather_group_profiles[
        weather_group_profiles["group_name"] == group_name
    ].set_index("service_category")["profile_value"]

    return temp.reindex(service_cols_weather).fillna(0)


rainy_weekday = get_weather_profile("rainy_weekday")
dry_weekday = get_weather_profile("dry_weekday")
rainy_weekend = get_weather_profile("rainy_weekend")
dry_weekend = get_weather_profile("dry_weekend")

rain_effect_weekday = rainy_weekday - dry_weekday
rain_effect_weekend = rainy_weekend - dry_weekend

rain_effect_df = pd.DataFrame({
    "service_category": service_cols_weather,
    "rain_effect_weekday": rain_effect_weekday.values,
    "rain_effect_weekend": rain_effect_weekend.values
})

rain_effect_df["mean_rain_effect"] = (
    rain_effect_df["rain_effect_weekday"] +
    rain_effect_df["rain_effect_weekend"]
) / 2

rain_effect_df = rain_effect_df.sort_values(
    "mean_rain_effect",
    ascending=False
)

output_csv = os.path.join(
    OUTPUT_DIR,
    "validation_weather_rain_effect_service_profile.csv"
)

rain_effect_df.to_csv(output_csv, index=False)

print("Saved:", output_csv)
display(rain_effect_df)

In [ ]:
# =========================
# Plot rainy effect
# =========================

plot_df = rain_effect_df.sort_values("mean_rain_effect")

plt.figure(figsize=(9, 6))

plt.barh(
    plot_df["service_category"],
    plot_df["rain_effect_weekday"],
    alpha=0.7,
    label="Rain effect on weekday"
)

plt.barh(
    plot_df["service_category"],
    plot_df["rain_effect_weekend"],
    alpha=0.5,
    label="Rain effect on weekend"
)

plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Rainy profile minus dry profile")
plt.ylabel("Service category")
plt.title("Weather Validation: Rainy vs Dry Service Profile Difference")
plt.legend()
plt.tight_layout()

output_fig = os.path.join(
    OUTPUT_DIR,
    "validation_weather_rainy_vs_dry_service_profile_difference.png"
)

plt.savefig(output_fig, dpi=300)
plt.show()

print("Saved:", output_fig)

In [ ]:
# =========================
# Cold vs mild profile differences
# =========================

cold_weekday = get_weather_profile("cold_weekday")
mild_weekday = get_weather_profile("mild_weekday")
cold_weekend = get_weather_profile("cold_weekend")
mild_weekend = get_weather_profile("mild_weekend")

cold_effect_weekday = cold_weekday - mild_weekday
cold_effect_weekend = cold_weekend - mild_weekend

cold_effect_df = pd.DataFrame({
    "service_category": service_cols_weather,
    "cold_effect_weekday": cold_effect_weekday.values,
    "cold_effect_weekend": cold_effect_weekend.values
})

cold_effect_df["mean_cold_effect"] = (
    cold_effect_df["cold_effect_weekday"] +
    cold_effect_df["cold_effect_weekend"]
) / 2

cold_effect_df = cold_effect_df.sort_values(
    "mean_cold_effect",
    ascending=False
)

output_csv = os.path.join(
    OUTPUT_DIR,
    "validation_weather_cold_effect_service_profile.csv"
)

cold_effect_df.to_csv(output_csv, index=False)

print("Saved:", output_csv)
display(cold_effect_df)

In [ ]:
# =========================
# Plot cold effect
# =========================

plot_df = cold_effect_df.sort_values("mean_cold_effect")

plt.figure(figsize=(9, 6))

plt.barh(
    plot_df["service_category"],
    plot_df["cold_effect_weekday"],
    alpha=0.7,
    label="Cold effect on weekday"
)

plt.barh(
    plot_df["service_category"],
    plot_df["cold_effect_weekend"],
    alpha=0.5,
    label="Cold effect on weekend"
)

plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Cold profile minus mild profile")
plt.ylabel("Service category")
plt.title("Weather Validation: Cold vs Mild Service Profile Difference")
plt.legend()
plt.tight_layout()

output_fig = os.path.join(
    OUTPUT_DIR,
    "validation_weather_cold_vs_mild_service_profile_difference.png"
)

plt.savefig(output_fig, dpi=300)
plt.show()

print("Saved:", output_fig)

In [ ]:
# =========================
# Compare weather effect strength with weekday/weekend structure
# =========================

def l1_distance(profile_a, profile_b):
    return np.abs(profile_a - profile_b).sum()


# Weekday/weekend structure under the same weather
dry_weekday_weekend_diff = l1_distance(dry_weekend, dry_weekday)
rainy_weekday_weekend_diff = l1_distance(rainy_weekend, rainy_weekday)

mild_weekday_weekend_diff = l1_distance(mild_weekend, mild_weekday)
cold_weekday_weekend_diff = l1_distance(cold_weekend, cold_weekday)

# Weather effects within the same day type
rain_effect_weekday_strength = l1_distance(rainy_weekday, dry_weekday)
rain_effect_weekend_strength = l1_distance(rainy_weekend, dry_weekend)

cold_effect_weekday_strength = l1_distance(cold_weekday, mild_weekday)
cold_effect_weekend_strength = l1_distance(cold_weekend, mild_weekend)

weather_validation_summary = pd.DataFrame({
    "comparison": [
        "weekday_weekend_structure_under_dry",
        "weekday_weekend_structure_under_rainy",
        "weekday_weekend_structure_under_mild",
        "weekday_weekend_structure_under_cold",
        "rain_effect_on_weekday",
        "rain_effect_on_weekend",
        "cold_effect_on_weekday",
        "cold_effect_on_weekend"
    ],
    "profile_difference_l1": [
        dry_weekday_weekend_diff,
        rainy_weekday_weekend_diff,
        mild_weekday_weekend_diff,
        cold_weekday_weekend_diff,
        rain_effect_weekday_strength,
        rain_effect_weekend_strength,
        cold_effect_weekday_strength,
        cold_effect_weekend_strength
    ]
})

output_csv = os.path.join(
    OUTPUT_DIR,
    "validation_weather_effect_strength_summary.csv"
)

weather_validation_summary.to_csv(output_csv, index=False)

print("Saved:", output_csv)
display(weather_validation_summary)

In [ ]:
# =========================
# Plot weather effect strength summary
# =========================

plot_df = weather_validation_summary.sort_values("profile_difference_l1")

plt.figure(figsize=(10, 6))

plt.barh(
    plot_df["comparison"],
    plot_df["profile_difference_l1"]
)

plt.xlabel("Service-profile difference, L1 distance")
plt.ylabel("Comparison")
plt.title("Weather Validation: Strength of Weather Effects vs Weekday/Weekend Structure")
plt.tight_layout()

output_fig = os.path.join(
    OUTPUT_DIR,
    "validation_weather_effect_strength_summary.png"
)

plt.savefig(output_fig, dpi=300)
plt.show()

print("Saved:", output_fig)